# Classical AI Logical Reasoning System

Name: Hemant nishad (P26AI0005)
Dept : SAIDE, IITJ



## 1. Internal Variable Represetation

- `LogicTerm`: constant or variable.
- `Proposition`: predicate, arguments and polarity.
- `DefiniteClause`: premises imply a conclusion.


In [1]:
import json
import time
import re
import os
import heapq
from difflib import SequenceMatcher
from itertools import product
import pandas as pd

# =====================================================================
# 1. INTERNAL REPRESENTATION DATA STRUCTURES
# =====================================================================

class LogicTerm:
    """Represents a logical term: either a Variable (?x) or a Constant (bob)."""
    def __init__(self, name, is_variable=False):
        self.name = str(name).strip()
        self.is_variable = is_variable

    def __repr__(self):
        return f"?{self.name}" if self.is_variable else self.name

    def __eq__(self, other):
        return isinstance(other, LogicTerm) and self.name == other.name and self.is_variable == other.is_variable

    def __hash__(self):
        return hash((self.name, self.is_variable))


class Proposition:
    """Represents a First-Order or Propositional Atom: e.g., Smart(bob) or ~Plant(x)."""
    def __init__(self, pred_name, arg_terms=None, is_negated=False):
        self.pred_name = str(pred_name).strip().lower()
        self.arg_terms = tuple(arg_terms) if arg_terms else ()
        self.is_negated = bool(is_negated)

    def complement(self):
        return Proposition(self.pred_name, self.arg_terms, not self.is_negated)

    def is_fully_ground(self):
        return all(not arg.is_variable for arg in self.arg_terms)

    def apply_bindings(self, theta):
        new_args = [theta.get(arg, arg) for arg in self.arg_terms]
        return Proposition(self.pred_name, new_args, self.is_negated)

    def __repr__(self):
        prefix = "¬" if self.is_negated else ""
        if not self.arg_terms:
            return f"{prefix}{self.pred_name}"
        args_str = ", ".join( repr(a) for a in self.arg_terms )
        return f"{prefix}{self.pred_name}({args_str})"

    def __eq__(self, other):
        return (isinstance(other, Proposition) and
                self.pred_name == other.pred_name and
                self.arg_terms == other.arg_terms and
                self.is_negated == other.is_negated)

    def __hash__(self):
        return hash((self.pred_name, self.arg_terms, self.is_negated))


class DefiniteClause:
    """Represents a definite/general rule: premise_1 ^ ... ^ premise_k -> conclusion."""
    def __init__(self, antecedents, consequent, source_text=""):
        self.antecedents = list(antecedents)
        self.consequent = consequent
        self.source_text = source_text

    def __repr__(self):
        hyps = " ∧ ".join(repr(p) for p in self.antecedents)
        return f"({hyps} → {repr(self.consequent)})"

## 2. Unification and Rule Reasoning

- **`unify_propositions`** matches logical atoms using consistent variable substitutions.
- **`infer_forward(goal)`** applies rules until the goal is derived or no new facts appear.
- **`infer_backward(goal)`** traces the goal back to supporting facts and rules while avoiding cycles.
- **`astar_derivation_search(goal)`** searches for proofs using unresolved subgoals as a heuristic.

Proof steps are recorded in `engine.proof_log`.

In [2]:
# =====================================================================
# 2. UNIFICATION, FORWARD CHAINING, BACKWARD CHAINING & A* SEARCH
# =====================================================================

def unify_propositions(prop1, prop2, bindings=None):
    """Classical Robinson Unification for two First-Order Logic atoms."""
    if bindings is None:
        bindings = {}
    if prop1.pred_name != prop2.pred_name or prop1.is_negated != prop2.is_negated:
        return None
    if len(prop1.arg_terms) != len(prop2.arg_terms):
        return None

    bindings = dict(bindings)
    for t1, t2 in zip(prop1.arg_terms, prop2.arg_terms):
        t1 = bindings.get(t1, t1)
        t2 = bindings.get(t2, t2)
        if t1 == t2:
            continue
        elif t1.is_variable:
            bindings[t1] = t2
        elif t2.is_variable:
            bindings[t2] = t1
        else:
            return None
    return bindings


class HornInferenceEngine:
    """
    Implements Classical AI Reasoning for Rule-Based KBs:
    1. Forward Chaining (Data-driven fixed-point inference)
    2. Backward Chaining (Goal-driven DFS with unification & cycle detection)
    3. A* Search (Heuristic shortest proof-tree search)
    """
    def __init__(self, fact_store=None, rule_list=None):
        self.fact_store = set(fact_store) if fact_store else set()
        self.rule_list = list(rule_list) if rule_list else []
        self.proof_log = []

    def infer_forward(self, target_prop):
        """Classical Forward Chaining with Unification."""
        known_set = set(self.fact_store)
        self.proof_log = [f"Initial Facts: {known_set}"]
        if target_prop in known_set:
            self.proof_log.append(f"Goal {target_prop} already in initial facts.")
            return True

        changed_flag = True
        while changed_flag:
            changed_flag = False
            for horn_clause in self.rule_list:
                # Match rule premises against current known ground facts
                binding_sets = self._match_body(horn_clause.antecedents, known_set, {})
                for bindings in binding_sets:
                    new_fact = horn_clause.consequent.apply_bindings(bindings)
                    if new_fact not in known_set:
                        known_set.add(new_fact)
                        changed_flag = True
                        body_text = " ∧ ".join(repr(p.apply_bindings(bindings)) for p in horn_clause.antecedents)
                        self.proof_log.append(f"Derived {new_fact} from {body_text} via {horn_clause}")
                        if new_fact == target_prop:
                            return True
        return target_prop in known_set

    def _match_body(self, antecedents, known_set, bindings):
        if not antecedents:
            return [bindings]
        head_goal = antecedents[0].apply_bindings(bindings)
        tail_goals = antecedents[1:]
        matches = []
        for fact in known_set:
            next_bindings = unify_propositions(head_goal, fact, bindings)
            if next_bindings is not None:
                matches.extend(self._match_body(tail_goals, known_set, next_bindings))
        return matches

    def infer_backward(self, target_prop, seen_nodes=None):
        """Classical Goal-Driven Backward Chaining (DFS) with cycle detection."""
        if seen_nodes is None:
            seen_nodes = set()
        if target_prop in self.fact_store:
            return True
        if target_prop in seen_nodes:
            return False
        seen_nodes.add(target_prop)

        for horn_clause in self.rule_list:
            bindings = unify_propositions(horn_clause.consequent, target_prop, {})
            if bindings is not None:
                subgoal_list = [p.apply_bindings(bindings) for p in horn_clause.antecedents]
                if self._prove_subgoals(subgoal_list, seen_nodes.copy()):
                    return True
        return False

    def _prove_subgoals(self, subgoal_list, seen_nodes):
        if not subgoal_list:
            return True
        head_goal = subgoal_list[0]
        tail_goals = subgoal_list[1:]
        if head_goal.is_fully_ground():
            if self.infer_backward(head_goal, seen_nodes.copy()):
                return self._prove_subgoals(tail_goals, seen_nodes)
            return False
        else:
            # Instantiate unbound variables using known constants
            const_pool = {arg for f in self.fact_store for arg in f.arg_terms if not arg.is_variable}
            for cand_const in const_pool:
                free_var = next(a for a in head_goal.arg_terms if a.is_variable)
                bindings = {free_var: cand_const}
                if self.infer_backward(head_goal.apply_bindings(bindings), seen_nodes.copy()):
                    if self._prove_subgoals([g.apply_bindings(bindings) for g in tail_goals], seen_nodes):
                        return True
            return False

    def astar_derivation_search(self, target_prop):
        """
        A* Heuristic Search to find the minimum-step derivation of goal_atom.
        State: frozenset of subgoals remaining to be proved.
        g(n): number of rule applications so far.
        h(n): number of remaining unproven subgoals not in self.fact_store.
        """
        initial_unproven = frozenset([] if target_prop in self.fact_store else [target_prop])
        if not initial_unproven:
            return True, 0

        # Priority Queue: (f = g + h, g, state_frozenset)
        frontier = [(len(initial_unproven), 0, initial_unproven)]
        seen_nodes = {initial_unproven: 0}

        while frontier:
            f, g, open_goals = heapq.heappop(frontier)
            if not open_goals:
                return True, g

            # Select first unproven goal to expand
            picked_goal = next(iter(open_goals))
            other_goals = set(open_goals) - {picked_goal}

            for horn_clause in self.rule_list:
                bindings = unify_propositions(horn_clause.consequent, picked_goal, {})
                if bindings is not None:
                    open_subgoals = {
                        p.apply_bindings(bindings)
                        for p in horn_clause.antecedents
                        if p.apply_bindings(bindings) not in self.fact_store
                    }
                    # Ground any unbound variables if needed
                    if all(sg.is_fully_ground() for sg in open_subgoals):
                        succ_state = frozenset(other_goals | open_subgoals)
                        new_g = g + 1
                        if succ_state not in seen_nodes or new_g < seen_nodes[succ_state]:
                            seen_nodes[succ_state] = new_g
                            h = len(succ_state)
                            heapq.heappush(frontier, (new_g + h, new_g, succ_state))
        return False, -1

## 3. Counterfactual Reasoning

The CounterBench module uses interventions (`do(X = x)`) to modify variables and evaluate their effects on the target.

- `A causes B`: \(B := A\)
- `A and B cause C`: \(C := A \land B\)
- `A or not B causes C`: \(C := A \lor \neg B\)

Observed root variables retain their values, while unobserved root variables default to `False`. The equations are evaluated repeatedly until the values stabilize.

In [3]:
# =====================================================================
# 3. STRUCTURAL CAUSAL MODEL & COUNTERFACTUAL ENGINE (COUNTERBENCH)
# =====================================================================

class StructuralModelSolver:
    """
    Classical Rung-3 Counterfactual Reasoning Engine using Pearl's 3-Step Procedure:
    1. Abduction: Fix observed exogenous variables ('We observed ...') to their observed values.
    2. Action: Apply do(X = x) counterfactual interventions to structural equations.
    3. Prediction: Forward-propagate structural assignments to evaluate target query.
    """
    def __init__(self):
        # Maps target_var -> list of (op_type, [(var, is_negated), ...], target_negated)
        self.structural_eqs = {}
        self.variable_set = set()
        self.observed_values = {}
        self.intervention_root = None

    @staticmethod
    def _normalize_literal(raw_tok):
        raw_tok = raw_tok.strip().strip(".,;").lower()
        raw_tok = re.sub(r"^(?:we know that|and|together)\s+", "", raw_tok).strip()
        if raw_tok.startswith("not "):
            return raw_tok[4:].strip(), True
        return raw_tok, False

    def parse_causal_spec(self, info_text, query_text):
        # 1. Extract observations if present
        obs_m = re.search(r"We observed\s+([^\.]+)", info_text, re.IGNORECASE)
        causal_body = info_text
        if obs_m:
            obs_text = obs_m.group(1).strip()
            causal_body = info_text[:obs_m.start()].strip()
            for item in re.split(r"\s+and\s+|,\s*", obs_text):
                if item.strip():
                    v, neg = self._normalize_literal(item)
                    if v:
                        self.observed_values[v] = not neg
                        self.variable_set.add(v)

        # 2. Extract causal equations from causal_text
        causal_body = re.sub(r"^We know that\s+", "", causal_body, flags=re.IGNORECASE).strip().rstrip(".")
        raw_parts = [s.strip() for s in causal_body.split(",") if s.strip()]

        # Reconstruct clauses where a single 'causes' has comma-separated targets
        equation_clauses = []
        prev_lhs = None
        for segment in raw_parts:
            if re.search(r"\bcauses?\b", segment, re.IGNORECASE):
                equation_clauses.append(segment)
                m = re.split(r"\bcauses?\b", segment, maxsplit=1, flags=re.IGNORECASE)
                prev_lhs = m[0].strip()
            elif prev_lhs is not None:
                # Continuation of RHS targets from previous LHS (e.g., "Sline, Frim, and Qado")
                rhs_clean = re.sub(r"^and\s+", "", segment, flags=re.IGNORECASE).strip()
                equation_clauses.append(f"{prev_lhs} causes {rhs_clean}")

        for equation_clause in equation_clauses:
            equation_clause = re.sub(r"^and\s+", "", equation_clause, flags=re.IGNORECASE).strip()
            parts = re.split(r"\s+together\s+causes?\s+|\s+causes?\s+", equation_clause, maxsplit=1, flags=re.IGNORECASE)
            if len(parts) != 2:
                continue
            lhs_text, rhs_text = parts[0].strip(), parts[1].strip()
            lhs_text = re.sub(r"\s+together$", "", lhs_text, flags=re.IGNORECASE).strip()

            # Determine LHS operator (AND vs OR)
            if re.search(r"\s+or\s+", lhs_text, re.IGNORECASE):
                op = "OR"
                lhs_parts = re.split(r"\s+or\s+", lhs_text, flags=re.IGNORECASE)
            else:
                op = "AND"
                lhs_parts = re.split(r"\s+and\s+|,\s*", lhs_text, flags=re.IGNORECASE)

            lhs_lits = []
            for tok_text in lhs_parts:
                if tok_text.strip():
                    v, neg = self._normalize_literal(tok_text)
                    lhs_lits.append((v, neg))
                    self.variable_set.add(v)

            # Parse RHS targets (may contain "A and B")
            rhs_parts = re.split(r"\s+and\s+", rhs_text, flags=re.IGNORECASE)
            for rhs_piece in rhs_parts:
                if rhs_piece.strip():
                    eq_target, tneg = self._normalize_literal(rhs_piece)
                    self.variable_set.add(eq_target)
                    self.structural_eqs.setdefault(eq_target, []).append((op, lhs_lits, tneg))

        # 3. Parse Question for Interventions and Target
        do_map = {}
        query_var = None

        # Pattern A: "Would X occur if Y instead of Z?" or "Would X occur if Y and W?"
        m1 = re.search(r"Would\s+([A-Za-z0-9_]+)\s+occur\s+if\s+(.+?)(?:\s+instead of\s+([A-Za-z0-9_]+))?\?", query_text, re.IGNORECASE)
        if m1:
            query_var = m1.group(1).strip().lower()
            do_text = m1.group(2).strip()
            if m1.group(3):
                self.intervention_root = m1.group(3).strip().lower()
            for do_part in re.split(r"\s+and\s+", do_text, flags=re.IGNORECASE):
                v, neg = self._normalize_literal(do_part)
                do_map[v] = not neg
                self.variable_set.add(v)
                if self.intervention_root is None:
                    self.intervention_root = v

        # Pattern B: "Assume X, and based on this assumption, further suppose Y. Would Z occur?"
        m2 = re.search(r"Assume\s+(.+?),\s*and based on this assumption,\s*further suppose\s+(.+?)\.\s*Would\s+([A-Za-z0-9_]+)\s+occur\?", query_text, re.IGNORECASE)
        if m2:
            v1, neg1 = self._normalize_literal(m2.group(1))
            v2, neg2 = self._normalize_literal(m2.group(2))
            query_var = m2.group(3).strip().lower()
            do_map[v1] = not neg1
            do_map[v2] = not neg2
            self.intervention_root = v1
            self.variable_set.update([v1, v2, query_var])

        return do_map, query_var

    def _propagate_equations(self, fixed_vals):
        """Forward-propagates structural equations to a fixed point."""
        world = {v: False for v in self.variable_set}
        world.update(fixed_vals)

        for _ in range(len(self.variable_set) + 2):
            changed_flag = False
            for eq_target, eq_entries in self.structural_eqs.items():
                if eq_target in fixed_vals:
                    continue
                # Evaluate rules for target variable tv
                truth_val = False
                for op, lit_list, tneg in eq_entries:
                    if op == "AND":
                        lhs_truth = all((not world[v] if neg else world[v]) for v, neg in lit_list)
                    else:
                        lhs_truth = any((not world[v] if neg else world[v]) for v, neg in lit_list)
                    rule_truth = (not lhs_truth) if tneg else lhs_truth
                    truth_val = truth_val or rule_truth
                if world[eq_target] != truth_val:
                    world[eq_target] = truth_val
                    changed_flag = True
            if not changed_flag:
                break
        return world

    def compute_counterfactual(self, info_text, query_text):
        do_map, query_var = self.parse_causal_spec(info_text, query_text)
        if not query_var:
            return False

        # Step 1: Abduction - Identify exogenous variables (no incoming causal equations)
        exog_vars = [v for v in self.variable_set if v not in self.structural_eqs]
        base_vals = {}

        # Any directly observed exogenous variable is fixed to its observed value
        for v, obs_val in self.observed_values.items():
            if v in exog_vars:
                base_vals[v] = obs_val

        # Step 2: Action - Apply do(interventions)
        cf_vals = dict(base_vals)
        cf_vals.update(do_map)

        # Step 3: Prediction - Evaluate counterfactual world
        cf_world = self._propagate_equations(cf_vals)
        return bool(cf_world.get(query_var, False))

## 4. FOLIO: Bounded DPLL-Based Entailment

The prover checks whether `KB ∧ ¬Conclusion` is unsatisfiable to determine entailment.

**Processing steps:**
1. Parse first-order logic statements.
2. Align unmatched vocabulary.
3. Ground quantifiers using up to three constants, or two for nested quantifiers.
4. Convert formulas to CNF using Tseitin transformation.
5. Apply DPLL with unit propagation.

**Limitation:** DPLL explores at most 100 nodes. If this limit is reached, the result is treated as satisfiable, which may cause some valid entailments to be missed.

In [4]:
# =====================================================================
# 4. FAST FIRST-ORDER LOGIC PARSER & BOUNDED DPLL PROVER (FOLIO)
# =====================================================================

class FormulaNode:
    def __init__(self, node_kind, payload=None, subnodes=None, var=None):
        self.node_kind = node_kind          # 'ATOM', 'NOT', 'AND', 'OR', 'IMP', 'IFF', 'XOR', 'ALL', 'EXISTS'
        self.payload = payload            # (pred_name, (arg1, arg2, ...)) for ATOM
        self.subnodes = subnodes or []
        self.var = var


class FolEntailmentChecker:
    """
    Fast Classical FOL Theorem Prover for FOLIO:
    1. Parses Unicode FOL formulas into an AST.
    2. Aligns conclusion predicates/constants with KB vocabulary.
    3. Dynamically bounds Herbrand domain size by quantifier depth.
    4. Uses Memoized Tseitin CNF + Step-Bounded DPLL branching only on ground atoms.
    """
    @staticmethod
    def sanitize_formula_line(text_line):
        text_line = re.sub(r"(?:Never:|Simeng:|\[).*$", "", text_line).strip()
        text_line = text_line.replace("—>", "→").replace("->", "→").replace("↔", "↔").replace("⟷", "↔")
        text_line = re.sub(r"\b([A-Za-z0-9_.-]+)\s*=\s*([A-Za-z0-9_.-]+)\b", r"Equal(\1, \2)", text_line)
        open_n, close_n = text_line.count("("), text_line.count(")")
        if open_n > close_n:
            text_line += ")" * (open_n - close_n)
        elif close_n > open_n:
            while text_line.endswith(")") and text_line.count(")") > text_line.count("("):
                text_line = text_line[:-1]
        return text_line

    @classmethod
    def lex_formula(cls, src_text):
        src_text = cls.sanitize_formula_line(src_text)
        lex_pattern = re.compile(
            r"\s*(?:(∀|∃)\s*([a-zA-Z0-9_]+)|(→|↔|⊕|∧|∨|¬|\(|\)|,)|([A-Za-z0-9_.\u00C0-\u024F-]+))"
        )
        tok_list = []
        cursor = 0
        while cursor < len(src_text):
            mt = lex_pattern.match(src_text, cursor)
            if not mt:
                cursor += 1
                continue
            quant_tok, quant_name, sym_tok, ident_tok = mt.groups()
            if quant_tok:
                tok_list.append(("QUANT", (quant_tok, quant_name.lower())))
            elif sym_tok:
                tok_list.append((sym_tok, sym_tok))
            elif ident_tok:
                tok_list.append(("ID", ident_tok))
            cursor = mt.end()
        return tok_list

    @classmethod
    def build_formula_ast(cls, src_text):
        tok_list = cls.lex_formula(src_text)
        if not tok_list:
            return None
        cursor = [0]

        def peek_tok():
            return tok_list[cursor[0]] if cursor[0] < len(tok_list) else (None, None)

        def take_tok():
            tok = peek_tok()
            cursor[0] += 1
            return tok

        def parse_biconditional():
            lhs_node = parse_implication()
            while peek_tok()[0] == "↔":
                take_tok()
                lhs_node = FormulaNode("IFF", subnodes=[lhs_node, parse_implication()])
            return lhs_node

        def parse_implication():
            lhs_node = parse_xor_expr()
            while peek_tok()[0] == "→":
                take_tok()
                lhs_node = FormulaNode("IMP", subnodes=[lhs_node, parse_xor_expr()])
            return lhs_node

        def parse_xor_expr():
            lhs_node = parse_disjunction()
            while peek_tok()[0] == "⊕":
                take_tok()
                lhs_node = FormulaNode("XOR", subnodes=[lhs_node, parse_disjunction()])
            return lhs_node

        def parse_disjunction():
            lhs_node = parse_conjunction()
            while peek_tok()[0] == "∨":
                take_tok()
                lhs_node = FormulaNode("OR", subnodes=[lhs_node, parse_conjunction()])
            return lhs_node

        def parse_conjunction():
            lhs_node = parse_negation_or_atom()
            while peek_tok()[0] == "∧":
                take_tok()
                lhs_node = FormulaNode("AND", subnodes=[lhs_node, parse_negation_or_atom()])
            return lhs_node

        def parse_negation_or_atom():
            tag, tag_val = peek_tok()
            if tag == "¬":
                take_tok()
                return FormulaNode("NOT", subnodes=[parse_negation_or_atom()])
            if tag == "QUANT":
                take_tok()
                quant_sym, quant_var = tag_val
                return FormulaNode("ALL" if quant_sym == "∀" else "EXISTS", subnodes=[parse_negation_or_atom()], var=quant_var)
            if tag == "(":
                take_tok()
                node = parse_biconditional()
                if peek_tok()[0] == ")":
                    take_tok()
                return node
            if tag == "ID":
                take_tok()
                pred = tag_val.lower()
                arg_terms = []
                if peek_tok()[0] == "(":
                    take_tok()
                    while peek_tok()[0] not in (")", None):
                        if peek_tok()[0] == ",":
                            take_tok()
                            continue
                        arg_tag, arg_val = take_tok()
                        if arg_tag == "ID":
                            arg_terms.append(arg_val.lower())
                    if peek_tok()[0] == ")":
                        take_tok()
                return FormulaNode("ATOM", payload=(pred, tuple(arg_terms)))
            take_tok()
            return FormulaNode("ATOM", payload=("true_atom", ()))

        return parse_biconditional()

    @staticmethod
    def gather_vocabulary(nodes):
        pred_sigs = set()
        const_sigs = set()
        default_vars = {"x", "y", "z", "w", "u", "v"}

        def walk(n, bound_names):
            if n is None:
                return
            if n.node_kind in ("ALL", "EXISTS"):
                walk(n.subnodes[0], bound_names | {n.var})
            elif n.node_kind == "ATOM":
                p, arg_terms = n.payload
                pred_sigs.add((p, len(arg_terms)))
                for a in arg_terms:
                    if a not in bound_names and a not in default_vars:
                        const_sigs.add(a)
            else:
                for c in n.subnodes:
                    walk(c, bound_names)

        for node in nodes:
            walk(node, set())
        return pred_sigs, const_sigs

    @classmethod
    def align_to_vocabulary(cls, node, kb_pred_sig, kb_const_sig):
        if node is None:
            return None
        if node.node_kind == "ATOM":
            p, arg_terms = node.payload
            aligned_args = []
            for a in arg_terms:
                if a in kb_const_sig or a in {"x", "y", "z", "w", "u", "v"}:
                    aligned_args.append(a)
                else:
                    best_const, best_sc = a, 0.0
                    for kconst in sorted(kb_const_sig):
                        sim = SequenceMatcher(None, a, kconst).ratio()
                        if sim > best_sc:
                            best_const, best_sc = kconst, sim
                    aligned_args.append(best_const if best_sc >= 0.72 else a)

            if (p, len(aligned_args)) not in kb_pred_sig:
                best_pred, best_sc = p, 0.0
                for kpred, karity in sorted(kb_pred_sig):
                    if karity == len(aligned_args):
                        sim = SequenceMatcher(None, p, kpred).ratio()
                        if sim > best_sc:
                            best_pred, best_sc = kpred, sim
                if best_sc >= 0.68:
                    p = best_pred
            return FormulaNode("ATOM", payload=(p, tuple(aligned_args)))
        return FormulaNode(node.node_kind, node.payload, [cls.align_to_vocabulary(c, kb_pred_sig, kb_const_sig) for c in node.subnodes], node.var)

    @classmethod
    def expand_quantifiers(cls, node, herbrand, bind_env=None, quant_depth=0):
        """Grounding with dynamic domain cap to prevent nested quantifier blowup."""
        if bind_env is None:
            bind_env = {}
        if node is None:
            return None

        if node.node_kind == "ATOM":
            p, arg_terms = node.payload
            return ("ATOM", (p, tuple(bind_env.get(a, a) for a in arg_terms)))
        if node.node_kind == "NOT":
            return ("NOT", cls.expand_quantifiers(node.subnodes[0], herbrand, bind_env, quant_depth))
        if node.node_kind in ("AND", "OR", "IMP", "IFF", "XOR"):
            return (
                node.node_kind,
                cls.expand_quantifiers(node.subnodes[0], herbrand, bind_env, quant_depth),
                cls.expand_quantifiers(node.subnodes[1], herbrand, bind_env, quant_depth),
            )
        if node.node_kind in ("ALL", "EXISTS"):
            # Cap branching factor for nested quantifiers (depth >= 1 -> max 2 constants)
            active_consts = herbrand[:3] if quant_depth == 0 else herbrand[:2]
            inst_nodes = [
                cls.expand_quantifiers(node.subnodes[0], herbrand, {**bind_env, node.var: c}, quant_depth + 1)
                for c in active_consts
            ]
            join_op = "AND" if node.node_kind == "ALL" else "OR"
            acc = inst_nodes[0]
            for s in inst_nodes[1:]:
                acc = (join_op, acc, s)
            return acc
        return None

    @classmethod
    def to_tseitin_cnf(cls, expr):
        """Memoized Tseitin CNF conversion returning (clauses, atom_var_ids)."""
        cnf_clauses = []
        atom_ids = {}
        cache = {}
        id_counter = [1]

        def atom_index(atom):
            if atom not in atom_ids:
                atom_ids[atom] = id_counter[0]
                id_counter[0] += 1
            return atom_ids[atom]

        def fresh_var():
            v = id_counter[0]
            id_counter[0] += 1
            return v

        def enc(e):
            if e in cache:
                return cache[e]
            k = e[0]
            if k == "ATOM":
                acc = atom_index(e[1])
                cache[e] = acc
                return acc
            if k == "NOT":
                acc = -enc(e[1])
                cache[e] = acc
                return acc
            a = enc(e[1])
            b = enc(e[2])
            z = fresh_var()
            if k == "AND":
                cnf_clauses.extend([[-z, a], [-z, b], [z, -a, -b]])
            elif k == "OR":
                cnf_clauses.extend([[z, -a], [z, -b], [-z, a, b]])
            elif k == "IMP":
                cnf_clauses.extend([[z, a], [z, -b], [-z, -a, b]])
            elif k == "IFF":
                cnf_clauses.extend([[-z, -a, b], [-z, a, -b], [z, a, b], [z, -a, -b]])
            elif k == "XOR":
                cnf_clauses.extend([[-z, a, b], [-z, -a, -b], [z, -a, b], [z, a, -b]])
            cache[e] = z
            return z

        top_lit = enc(expr)
        cnf_clauses.append([top_lit])
        return cnf_clauses, set(atom_ids.values())

    @classmethod
    def dpll_search(cls, cnf_clauses, atom_vars, partial_model=None, budget=None):
        """Bounded DPLL SAT solver that branches strictly on ground atom variables."""
        if partial_model is None:
            partial_model = {}
        if budget is None:
            budget = [0]

        budget[0] += 1
        if budget[0] > 100:
            # Step budget reached: abort search and treat as satisfiable (non-entailed)
            return True

        # Unit Propagation
        changed_flag = True
        while changed_flag:
            changed_flag = False
            kept_clauses = []
            for c in cnf_clauses:
                open_lits = []
                satisfied = False
                for lit in c:
                    var = abs(lit)
                    if var in partial_model:
                        if partial_model[var] == (lit > 0):
                            satisfied = True
                            break
                    else:
                        open_lits.append(lit)
                if satisfied:
                    continue
                if not open_lits:
                    return False  # Empty clause derived (Contradiction!)
                if len(open_lits) == 1:
                    u_lit = open_lits[0]
                    u_var = abs(u_lit)
                    u_val = (u_lit > 0)
                    if u_var in partial_model and partial_model[u_var] != u_val:
                        return False
                    partial_model[u_var] = u_val
                    changed_flag = True
                else:
                    kept_clauses.append(open_lits)
            cnf_clauses = kept_clauses

        if not cnf_clauses:
            return True

        # Prefer branching on unassigned Ground Atom variables (Tseitin gates propagate automatically)
        branch_lit = None
        for c in cnf_clauses:
            for lit in c:
                if abs(lit) in atom_vars and abs(lit) not in partial_model:
                    branch_lit = lit
                    break
            if branch_lit is not None:
                break

        if branch_lit is None:
            branch_lit = cnf_clauses[0][0]

        var = abs(branch_lit)
        for polarity in (branch_lit > 0, branch_lit < 0):
            next_model = dict(partial_model)
            next_model[var] = polarity
            if cls.dpll_search(cnf_clauses, atom_vars, next_model, budget):
                return True
        return False

    @classmethod
    def check_entailment(cls, premises_fol_str, conclusion_fol_str):
        if not premises_fol_str or not conclusion_fol_str:
            return False
        prem_lines = [l.strip() for l in premises_fol_str.split("\n") if l.strip()]
        prem_nodes = []
        for l in prem_lines:
            try:
                ast = cls.build_formula_ast(l)
                if ast:
                    prem_nodes.append(ast)
            except Exception:
                pass

        try:
            conc_node = cls.build_formula_ast(conclusion_fol_str)
        except Exception:
            return False
        if not prem_nodes or conc_node is None:
            return False

        kb_pred_sig, kb_const_sig = cls.gather_vocabulary(prem_nodes)
        conc_node = cls.align_to_vocabulary(conc_node, kb_pred_sig, kb_const_sig)
        _, aligned_conc_consts = cls.gather_vocabulary([conc_node])

        # Prioritize constants mentioned in the conclusion first, then KB constants (max 3)
        ordered_consts = sorted(aligned_conc_consts) + [c for c in sorted(kb_const_sig) if c not in aligned_conc_consts]
        herbrand = ordered_consts[:3] if ordered_consts else ["c_default"]

        ground_body = [cls.expand_quantifiers(p, herbrand) for p in prem_nodes]
        ground_neg = ("NOT", cls.expand_quantifiers(conc_node, herbrand))

        combined = ground_body[0]
        for gp in ground_body[1:]:
            combined = ("AND", combined, gp)

        refutation_formula = ("AND", combined, ground_neg)
        cnf_clauses, atom_vars = cls.to_tseitin_cnf(refutation_formula)

        # If KB ∧ ¬Conclusion is UNSAT, check that KB itself is satisfiable
        if not cls.dpll_search(cnf_clauses, atom_vars):
            kb_clauses, kb_atom_vars = cls.to_tseitin_cnf(combined)
            if cls.dpll_search(kb_clauses, kb_atom_vars):
                return True
        return False

## 5. Natural-Language Parsing

`TextKnowledgeParser` converts supported sentence patterns into facts and rules.

- `parse_facts_and_rules`: reads `Facts:` and `Rules:` and handles five conditional phrasings.
- `parse_natural_kb`: handles simple facts, conditional rules and universal statements.

Text is normalized before parsing; variables are shown as `?x`.

In [5]:
# =====================================================================
# 5. NATURAL LANGUAGE PARSERS (RULETAKER & GENERAL NL KBs)
# =====================================================================

class TextKnowledgeParser:
    """
    Converts Natural Language Knowledge Bases + Queries into Internal Representation
    (Facts + HornRules) for RuleTaker and general NL syllogisms.
    """
    CONDITIONAL_PATTERNS = [
        re.compile(r"^(?:based on|assuming|when|if|given that)\s+(.+?),\s*(?:it is evident that|we can say|we can conclude|then|it follows that)\s+(.+?)\.?$", re.IGNORECASE),
    ]

    @staticmethod
    def canonicalize_text(text):
        text = text.strip().rstrip(".!?").strip().lower()
        text = re.sub(r"\s+", " ", text)
        return text

    @classmethod
    def parse_facts_and_rules(cls, ctx_text, stmt_text):
        fact_store = set()
        rule_list = []

        section = "facts"
        for raw_line in ctx_text.split("\n"):
            line_text = raw_line.strip()
            if not line_text:
                continue
            if line_text.lower().startswith("facts:"):
                section = "facts"
                continue
            if line_text.lower().startswith("rules:"):
                section = "rules"
                continue

            if section == "facts":
                prop_text = cls.canonicalize_text(line_text)
                if prop_text:
                    fact_store.add(Proposition(prop_text))
            else:
                hit = False
                for cond_pat in cls.CONDITIONAL_PATTERNS:
                    m = cond_pat.match(line_text)
                    if m:
                        body_str = cls.canonicalize_text(m.group(1))
                        head_str = cls.canonicalize_text(m.group(2))
                        body_list = [Proposition(cls.canonicalize_text(pname)) for pname in re.split(r"\s+and\s+", body_str)]
                        rule_list.append(DefiniteClause(body_list, Proposition(head_str), source_text=line_text))
                        hit = True
                        break
                if not hit:
                    prop_text = cls.canonicalize_text(line_text)
                    if prop_text:
                        fact_store.add(Proposition(prop_text))

        query_prop = Proposition(cls.canonicalize_text(stmt_text))
        return HornInferenceEngine(fact_store, rule_list), query_prop

    @classmethod
    def parse_natural_kb(cls, ctx_text, query_text):
        """
        Parses general natural language KBs such as the Assignment PDF example:
        'Bob is young. Bob is kind. If someone is young, then they are smart.
         If someone is smart and kind, then they are successful.'
        """
        # If formatted with 'Facts:' and 'Rules:', delegate to RuleTaker parser
        if "Facts:" in ctx_text and "Rules:" in ctx_text:
            return cls.parse_facts_and_rules(ctx_text, query_text)

        fact_store = set()
        rule_list = []
        sentence_list = [s.strip() for s in re.split(r"[.\n]+", ctx_text) if s.strip()]

        for sentence in sentence_list:
            # Pattern 1: "If someone is P1 and P2, then they are Q"
            m_forall = re.match(
                r"^If\s+(?:someone|a person|anyone|something)\s+is\s+(.+?),\s*then\s+(?:they|he|she|it)\s+(?:are|is)\s+(.+)$",
                sentence, re.IGNORECASE
            )
            if m_forall:
                cond_names = [c.strip().lower() for c in re.split(r"\s+and\s+", m_forall.group(1))]
                head_name = m_forall.group(2).strip().lower()
                var_term = LogicTerm("x", is_variable=True)
                body_atoms = [Proposition(c, [var_term]) for c in cond_names]
                rule_list.append(DefiniteClause(body_atoms, Proposition(head_name, [var_term]), source_text=sentence))
                continue

            # Pattern 2: "All P are Q" / "Every P is Q"
            m_every = re.match(r"^(?:All|Every|Each)\s+([a-zA-Z\s]+?)\s+(?:are|is)\s+([a-zA-Z\s]+)$", sentence, re.IGNORECASE)
            if m_every:
                pname = m_every.group(1).strip().lower().rstrip("s")
                q = m_every.group(2).strip().lower().rstrip("s")
                var_term = LogicTerm("x", is_variable=True)
                rule_list.append(DefiniteClause([Proposition(pname, [var_term])], Proposition(q, [var_term]), source_text=sentence))
                continue

            # Pattern 3: Propositional conditional "If A, then B"
            for cond_pat in cls.CONDITIONAL_PATTERNS:
                m_cond = cond_pat.match(sentence)
                if m_cond:
                    ante_str = cls.canonicalize_text(m_cond.group(1))
                    cons_str = cls.canonicalize_text(m_cond.group(2))
                    rule_list.append(DefiniteClause([Proposition(ante_str)], Proposition(cons_str), source_text=sentence))
                    break
            else:
                # Pattern 4: Ground fact "Bob is young"
                m_ground = re.match(r"^([A-Za-z0-9_]+)\s+is\s+(?:a\s+|an\s+)?(.+)$", sentence.strip(), re.IGNORECASE)
                if m_ground:
                    entity_term = LogicTerm(m_ground.group(1).lower(), is_variable=False)
                    pred_names = [pname.strip().lower().rstrip("s") for pname in re.split(r"\s+and\s+", m_ground.group(2))]
                    for pname in pred_names:
                        fact_store.add(Proposition(pname, [entity_term]))
                else:
                    fact_store.add(Proposition(cls.canonicalize_text(sentence)))

        # Parse Query
        query_clean = query_text.strip().rstrip(".!?")
        m_query = re.match(r"^([A-Za-z0-9_]+)\s+is\s+(?:a\s+|an\s+)?(.+)$", query_clean, re.IGNORECASE)
        if m_query:
            query_prop = Proposition(m_query.group(2).strip().lower().rstrip("s"), [LogicTerm(m_query.group(1).lower(), is_variable=False)])
        else:
            query_prop = Proposition(cls.canonicalize_text(query_clean))

        return HornInferenceEngine(fact_store, rule_list), query_prop

## 5b. FOLIO: English to First-Order Logic and Resolution

A rule-based parser converts the English premises to first-order formulas (quantifiers, connectives, relative clauses, conditionals). Formulas are converted to NNF, Skolemised and put in CNF; a set-of-support resolution prover refutes the negated conclusion. Premises that cannot be parsed are dropped, which can only lose entailments. The dataset's gold FOL annotations are used only when `USE_GOLD_FOL = True` (development).


In [6]:
from __future__ import annotations

from dataclasses import dataclass
from itertools import count, product
import heapq
import time
from itertools import count
import re
from typing import Callable


def _build_fol_engine():
    # --- logic/terms.py ---
    @dataclass(frozen=True, slots=True)
    class Var:
        name: str

        def __str__(self) -> str:
            return f"?{self.name}"


    @dataclass(frozen=True, slots=True)
    class Fn:
        name: str
        args: tuple[Term, ...]

        def __str__(self) -> str:
            return f"{self.name}({', '.join(map(str, self.args))})"


    Term = str | Var | Fn
    Fact = tuple[str, tuple[str, ...]]


    def is_ground_term(term: Term) -> bool:
        if isinstance(term, Var):
            return False
        if isinstance(term, Fn):
            return all(is_ground_term(a) for a in term.args)
        return True


    @dataclass(frozen=True, slots=True)
    class Atom:
        pred: str
        args: tuple[Term, ...]

        @property
        def is_ground(self) -> bool:
            return all(is_ground_term(a) for a in self.args)

        def __str__(self) -> str:
            return f"{self.pred}({', '.join(map(str, self.args))})"


    @dataclass(frozen=True, slots=True)
    class Literal:
        atom: Atom
        positive: bool = True

        def __str__(self) -> str:
            return str(self.atom) if self.positive else f"not {self.atom}"


    @dataclass(frozen=True, slots=True)
    class Rule:
        head: Literal
        body: tuple[Literal, ...]

        def __str__(self) -> str:
            return f"{' & '.join(map(str, self.body))} -> {self.head}"


    # --- logic/unify.py ---
    Subst = dict[Var, Term]


    def walk(term: Term, subst: Subst) -> Term:
        while isinstance(term, Var) and term in subst:
            term = subst[term]
        return term


    def occurs(var: Var, term: Term, subst: Subst) -> bool:
        term = walk(term, subst)
        if term == var:
            return True
        if isinstance(term, Fn):
            return any(occurs(var, a, subst) for a in term.args)
        return False


    def unify_terms(a: Term, b: Term, subst: Subst) -> Subst | None:
        a, b = walk(a, subst), walk(b, subst)
        if a == b:
            return subst
        if isinstance(a, Var):
            return None if occurs(a, b, subst) else {**subst, a: b}
        if isinstance(b, Var):
            return None if occurs(b, a, subst) else {**subst, b: a}
        if isinstance(a, Fn) and isinstance(b, Fn) and a.name == b.name and len(a.args) == len(b.args):
            for x, y in zip(a.args, b.args):
                subst = unify_terms(x, y, subst)
                if subst is None:
                    return None
            return subst
        return None


    def unify(x: Atom, y: Atom, subst: Subst | None = None) -> Subst | None:
        if x.pred != y.pred or len(x.args) != len(y.args):
            return None
        subst = {} if subst is None else subst
        for a, b in zip(x.args, y.args):
            subst = unify_terms(a, b, subst)
            if subst is None:
                return None
        return subst


    def match_term(pattern: Term, target: Term, subst: Subst) -> Subst | None:
        if isinstance(pattern, Var):
            bound = subst.get(pattern)
            if bound is None:
                return {**subst, pattern: target}
            return subst if bound == target else None
        if isinstance(pattern, Fn):
            if not isinstance(target, Fn) or pattern.name != target.name or len(pattern.args) != len(target.args):
                return None
            for p, t in zip(pattern.args, target.args):
                subst = match_term(p, t, subst)
                if subst is None:
                    return None
            return subst
        return subst if pattern == target else None


    def match(pattern: tuple[Term, ...], ground: tuple[Term, ...], subst: Subst) -> Subst | None:
        if len(pattern) != len(ground):
            return None
        out = subst
        for p, g in zip(pattern, ground):
            if isinstance(p, Var):
                bound = out.get(p)
                if bound is None:
                    if out is subst:
                        out = dict(subst)
                    out[p] = g
                elif bound != g:
                    return None
            elif isinstance(p, str):
                if p != g:
                    return None
            else:
                out = match_term(p, g, out)
                if out is None:
                    return None
        return out


    def apply(term: Term, subst: Subst) -> Term:
        term = walk(term, subst)
        if isinstance(term, Fn):
            return Fn(term.name, tuple(apply(a, subst) for a in term.args))
        return term


    def substitute(atom: Atom, subst: Subst) -> Atom:
        return Atom(atom.pred, tuple(apply(a, subst) for a in atom.args))


    # --- logic/formula.py ---
    @dataclass(frozen=True, slots=True)
    class Pred:
        name: str
        args: tuple[Term, ...]


    @dataclass(frozen=True, slots=True)
    class Not:
        f: Formula


    @dataclass(frozen=True, slots=True)
    class And:
        fs: tuple[Formula, ...]


    @dataclass(frozen=True, slots=True)
    class Or:
        fs: tuple[Formula, ...]


    @dataclass(frozen=True, slots=True)
    class Imp:
        a: Formula
        b: Formula


    @dataclass(frozen=True, slots=True)
    class Iff:
        a: Formula
        b: Formula


    @dataclass(frozen=True, slots=True)
    class Xor:
        a: Formula
        b: Formula


    @dataclass(frozen=True, slots=True)
    class Forall:
        var: Var
        body: Formula


    @dataclass(frozen=True, slots=True)
    class Exists:
        var: Var
        body: Formula


    Formula = Pred | Not | And | Or | Imp | Iff | Xor | Forall | Exists


    def show(f: Formula) -> str:
        if isinstance(f, Pred):
            return f"{f.name}({', '.join(map(str, f.args))})"
        if isinstance(f, Not):
            return f"¬{show(f.f)}"
        if isinstance(f, And):
            return "(" + " ∧ ".join(map(show, f.fs)) + ")"
        if isinstance(f, Or):
            return "(" + " ∨ ".join(map(show, f.fs)) + ")"
        if isinstance(f, Imp):
            return f"({show(f.a)} → {show(f.b)})"
        if isinstance(f, Iff):
            return f"({show(f.a)} ↔ {show(f.b)})"
        if isinstance(f, Xor):
            return f"({show(f.a)} ⊕ {show(f.b)})"
        if isinstance(f, Forall):
            return f"∀{f.var.name} {show(f.body)}"
        return f"∃{f.var.name} {show(f.body)}"


    # --- logic/cnf.py ---
    Clause = tuple[Literal, ...]

    MAX_CLAUSES = 256


    class TooLarge(Exception):
        pass


    def nnf(f: Formula, positive: bool = True) -> Formula:
        if isinstance(f, Pred):
            return f if positive else Not(f)
        if isinstance(f, Not):
            return nnf(f.f, not positive)
        if isinstance(f, And):
            parts = tuple(nnf(g, positive) for g in f.fs)
            return And(parts) if positive else Or(parts)
        if isinstance(f, Or):
            parts = tuple(nnf(g, positive) for g in f.fs)
            return Or(parts) if positive else And(parts)
        if isinstance(f, Imp):
            return nnf(Or((Not(f.a), f.b)), positive)
        if isinstance(f, Iff):
            return nnf(And((Imp(f.a, f.b), Imp(f.b, f.a))), positive)
        if isinstance(f, Xor):
            return nnf(And((Or((f.a, f.b)), Not(And((f.a, f.b))))), positive)
        if isinstance(f, Forall):
            body = nnf(f.body, positive)
            return Forall(f.var, body) if positive else Exists(f.var, body)
        body = nnf(f.body, positive)
        return Exists(f.var, body) if positive else Forall(f.var, body)


    def _subst_term(term: Term, env: dict[Var, Term]) -> Term:
        if isinstance(term, Var):
            return env.get(term, term)
        if isinstance(term, Fn):
            return Fn(term.name, tuple(_subst_term(a, env) for a in term.args))
        return term


    class _Skolemizer:
        def __init__(self, tag: str):
            self.tag = tag
            self.ids = count()

        def run(self, f: Formula, env: dict[Var, Term], universals: tuple[Var, ...]) -> Formula:
            if isinstance(f, Pred):
                return Pred(f.name, tuple(_subst_term(a, env) for a in f.args))
            if isinstance(f, Not):
                return Not(self.run(f.f, env, universals))
            if isinstance(f, (And, Or)):
                return type(f)(tuple(self.run(g, env, universals) for g in f.fs))
            n = next(self.ids)
            if isinstance(f, Forall):
                fresh = Var(f"{f.var.name}_{self.tag}{n}")
                return self.run(f.body, {**env, f.var: fresh}, universals + (fresh,))
            skolem = Fn(f"sk{self.tag}{n}", universals) if universals else f"sk{self.tag}{n}"
            return self.run(f.body, {**env, f.var: skolem}, universals)


    def _literals(f: Formula) -> list[list[Literal]]:
        if isinstance(f, Pred):
            return [[Literal(Atom(f.name, f.args), True)]]
        if isinstance(f, Not):
            p = f.f
            return [[Literal(Atom(p.name, p.args), False)]]
        if isinstance(f, And):
            out: list[list[Literal]] = []
            for g in f.fs:
                out.extend(_literals(g))
                if len(out) > MAX_CLAUSES:
                    raise TooLarge
            return out
        parts = [_literals(g) for g in f.fs]
        size = 1
        for p in parts:
            size *= len(p)
        if size > MAX_CLAUSES:
            raise TooLarge
        return [[lit for clause in combo for lit in clause] for combo in product(*parts)]


    def _tidy(literals: list[Literal]) -> Clause | None:
        seen: dict[Literal, None] = {}
        for lit in literals:
            seen[lit] = None
        for lit in seen:
            if Literal(lit.atom, not lit.positive) in seen:
                return None
        return tuple(seen)


    def to_clauses(f: Formula, tag: str = "") -> list[Clause]:
        skolemized = _Skolemizer(tag).run(nnf(f), {}, ())
        clauses = []
        for raw in _literals(skolemized):
            clause = _tidy(raw)
            if clause is not None:
                clauses.append(clause)
        return clauses


    # --- inference/resolution.py ---
    @dataclass
    class Result:
        proved: bool
        given: int = 0
        generated: int = 0
        reason: str = "saturated"
        steps: list | None = None


    def _size(term: Term) -> int:
        if isinstance(term, Fn):
            return 1 + sum(_size(a) for a in term.args)
        return 1


    def _weight(clause: Clause) -> int:
        return sum(1 + sum(_size(a) for a in lit.atom.args) for lit in clause)


    def _term_vars(term: Term, out: list[Var]) -> None:
        if isinstance(term, Var):
            out.append(term)
        elif isinstance(term, Fn):
            for a in term.args:
                _term_vars(a, out)


    def _rename_term(term: Term, mapping: dict[Var, Var]) -> Term:
        if isinstance(term, Var):
            return mapping[term]
        if isinstance(term, Fn):
            return Fn(term.name, tuple(_rename_term(a, mapping) for a in term.args))
        return term


    def _rename(clause: Clause, tag: str) -> Clause:
        mapping: dict[Var, Var] = {}
        found: list[Var] = []
        for lit in clause:
            for a in lit.atom.args:
                _term_vars(a, found)
        for v in found:
            if v not in mapping:
                mapping[v] = Var(f"{tag}{len(mapping)}")
        return tuple(
            Literal(Atom(l.atom.pred, tuple(_rename_term(a, mapping) for a in l.atom.args)), l.positive)
            for l in clause
        )


    def _normalize(clause: Clause) -> Clause:
        ordered = sorted(clause, key=lambda l: (l.atom.pred, l.positive, _size_key(l)))
        return _rename(tuple(ordered), "_")


    def _size_key(lit: Literal) -> str:
        return "".join("*" if isinstance(a, Var) else str(a) for a in lit.atom.args)


    def _tautology(clause: Clause) -> bool:
        keys = {(l.atom, l.positive) for l in clause}
        return any((a, not s) in keys for a, s in keys)


    def _dedupe(literals: list[Literal]) -> Clause:
        return tuple(dict.fromkeys(literals))


    def subsumes(general: Clause, specific: Clause) -> bool:
        if len(general) > len(specific):
            return False

        def go(i: int, subst: dict) -> bool:
            if i == len(general):
                return True
            lit = general[i]
            for other in specific:
                if other.positive != lit.positive or other.atom.pred != lit.atom.pred:
                    continue
                s: dict | None = subst
                for p, t in zip(lit.atom.args, other.atom.args):
                    s = match_term(p, t, s)
                    if s is None:
                        break
                if s is not None and go(i + 1, s):
                    return True
            return False

        return go(0, {})


    def _resolvents(c1: Clause, c2: Clause) -> list[Clause]:
        out = []
        for i, l1 in enumerate(c1):
            for j, l2 in enumerate(c2):
                if l1.positive == l2.positive or l1.atom.pred != l2.atom.pred:
                    continue
                subst = unify(l1.atom, l2.atom)
                if subst is None:
                    continue
                rest = [l for k, l in enumerate(c1) if k != i] + [l for k, l in enumerate(c2) if k != j]
                out.append(_dedupe([Literal(substitute(l.atom, subst), l.positive) for l in rest]))
        return out


    def _factors(clause: Clause) -> list[Clause]:
        out = []
        for i in range(len(clause)):
            for j in range(i + 1, len(clause)):
                a, b = clause[i], clause[j]
                if a.positive != b.positive or a.atom.pred != b.atom.pred:
                    continue
                subst = unify(a.atom, b.atom)
                if subst is not None:
                    out.append(_dedupe([Literal(substitute(l.atom, subst), l.positive) for l in clause]))
        return out


    def refute(
        usable: list[Clause],
        sos: list[Clause],
        max_given: int = 3000,
        max_generated: int = 40000,
        timeout: float = 1.0,
    ) -> Result:
        deadline = time.perf_counter() + timeout
        tiebreak = count()
        renames = count()
        passive: list[tuple[int, int, Clause]] = []
        seen: set[Clause] = set()
        kept_by_pred: dict[str, list[Clause]] = {}
        partners: list[Clause] = []

        def keep(clause: Clause) -> None:
            for lit in clause:
                kept_by_pred.setdefault(lit.atom.pred, []).append(clause)
            partners.append(clause)

        def redundant(clause: Clause) -> bool:
            if clause in seen:
                return True
            seen.add(clause)
            for lit in clause:
                for other in kept_by_pred.get(lit.atom.pred, ()):
                    if len(other) <= len(clause) and subsumes(other, clause):
                        return True
            return False

        for clause in usable:
            norm = _normalize(clause)
            if not norm:
                return Result(True, reason="inconsistent premises")
            seen.add(norm)
            keep(norm)
        for clause in sos:
            norm = _normalize(clause)
            if not norm:
                return Result(True, reason="empty clause")
            if norm not in seen:
                seen.add(norm)
                heapq.heappush(passive, (_weight(norm), next(tiebreak), norm))

        given = generated = 0
        while passive:
            if given >= max_given or generated >= max_generated:
                return Result(False, given, generated, "limit")
            if time.perf_counter() > deadline:
                return Result(False, given, generated, "timeout")
            _, _, current = heapq.heappop(passive)
            if any(len(o) <= len(current) and subsumes(o, current) for lit in current
                   for o in kept_by_pred.get(lit.atom.pred, ())):
                continue
            given += 1
            candidates = list(_factors(current))
            seen_partners: set[int] = set()
            for lit in current:
                for other in kept_by_pred.get(lit.atom.pred, ()):
                    if id(other) in seen_partners:
                        continue
                    seen_partners.add(id(other))
                    candidates.extend(_resolvents(current, _rename(other, f"r{next(renames)}_")))
            candidates.extend(_resolvents(current, _rename(current, f"r{next(renames)}_")))
            keep(current)
            for raw in candidates:
                generated += 1
                if not raw:
                    return Result(True, given, generated, "refuted")
                if _tautology(raw):
                    continue
                clause = _normalize(raw)
                if redundant(clause):
                    continue
                heapq.heappush(passive, (_weight(clause), next(tiebreak), clause))
        return Result(False, given, generated, "saturated")


    # --- inference/prover.py ---
    def premise_clauses(premises: list[Formula]) -> list[Clause]:
        clauses: list[Clause] = []
        for i, premise in enumerate(premises):
            try:
                clauses.extend(to_clauses(premise, tag=f"p{i}_"))
            except TooLarge:
                continue
        return clauses


    def entails(premises: list[Formula], query: Formula, **limits) -> Result:
        return prove(premise_clauses(premises), query, **limits)


    def prove(usable: list[Clause], query: Formula, **limits) -> Result:
        try:
            sos = to_clauses(Not(query), tag="q_")
        except TooLarge:
            return Result(False, reason="too large")
        return refute(usable, sos, **limits)


    # --- parsing/lexicon.py ---
    DETERMINERS = {"a", "an", "the"}
    COPULAS = {"is", "are", "was", "were", "be", "been", "being", "am"}
    DO = {"do", "does", "did"}
    HAVE = {"has", "have", "had"}
    MODALS = {"can", "could", "will", "would", "should", "may", "might", "must", "shall"}
    FILLERS = {"regularly", "often", "usually", "frequently", "really", "also", "currently", "typically",
               "generally", "just", "already", "actually", "still"}
    PREPOSITIONS = {"in", "on", "at", "of", "from", "by", "to", "with", "for", "near", "into", "about", "as",
                    "under", "over", "after", "before", "during", "between", "against", "through", "across",
                    "around", "within", "without", "than", "among", "upon", "via"}
    PRONOUNS = {"he", "she", "it", "they", "him", "her", "them"}
    GENERIC_NOUNS = {"person", "people", "thing", "one", "individual", "ones"}
    GENERIC_PRONOUN_QUANTIFIERS = {
        "someone": "exists", "somebody": "exists", "something": "exists",
        "everyone": "forall", "everybody": "forall", "everything": "forall",
        "anyone": "forall", "anybody": "forall", "anything": "forall",
        "nobody": "none", "nothing": "none",
    }
    QUANTIFIERS = {"all": "forall", "every": "forall", "each": "forall", "any": "forall",
                   "no": "none", "some": "exists", "several": "exists"}

    IRREGULAR_STEMS = {
        "people": "person", "men": "man", "women": "woman", "children": "child", "feet": "foot",
        "teeth": "tooth", "mice": "mouse", "geese": "goose", "wives": "wife", "lives": "life",
        "knives": "knife", "leaves": "leaf", "persons": "person",
    }

    IRREGULAR_PAST = {
        "ate": "eat", "wrote": "write", "drank": "drink", "went": "go", "saw": "see", "took": "take",
        "gave": "give", "made": "make", "won": "win", "lost": "lose", "led": "lead", "ran": "run",
        "came": "come", "sat": "sit", "stood": "stand", "told": "tell", "thought": "think",
        "brought": "bring", "built": "build", "bought": "buy", "caught": "catch", "fought": "fight",
        "held": "hold", "kept": "keep", "left": "leave", "met": "meet", "paid": "pay", "said": "say",
        "sold": "sell", "sent": "send", "spent": "spend", "taught": "teach", "understood": "understand",
        "wore": "wear", "began": "begin", "broke": "break", "chose": "choose", "drove": "drive",
        "fell": "fall", "flew": "fly", "forgot": "forget", "got": "get", "grew": "grow", "knew": "know",
        "rode": "ride", "rose": "rise", "sang": "sing", "spoke": "speak", "swam": "swim", "threw": "throw",
        "found": "find", "heard": "hear", "felt": "feel", "became": "become", "put": "put", "set": "set",
        "slept": "sleep", "stole": "steal", "shot": "shoot", "wrote": "write", "hid": "hide", "bit": "bite",
    }

    BASE_VERBS = """
    accept achieve act add admit affect agree allow announce appear apply argue arrive ask attack attend
    avoid bake base bear beat become begin believe belong bite break bring build buy call care carry catch
    cause celebrate change charge chase check choose claim clean climb close collect come compete complete
    compose conduct consider consist contain continue control cook cost count cover create cross cry cut
    dance deal decide defeat deliver depend describe design destroy develop die discover discuss draw dream
    drink drive drop eat earn edit employ enjoy enter establish exist expect explain face fail fall feed feel
    fight fill find finish fit fly focus follow form found gain get give go graduate grow guide handle happen
    hate head hear help hit hold hope host hunt identify imagine improve include increase indicate involve
    join jump keep kill kiss know lack land last lead learn leave lend let lie like listen live look lose
    love maintain make manage mean meet mention move name need notice obtain occur offer open operate order
    own participate pass pay perform pick place plan plant play point possess prefer prepare present press
    prevent produce promote protect prove provide publish pull push put qualify raise reach read realize
    receive recognize record reduce refer reflect refuse regret relate release remain remember remove repeat
    replace represent require resist respond rest result return reveal review ride ring rise rule run save
    say score search see seek seem sell send serve set share show sign sing sit sleep smell solve speak spend
    split stand start stay steal stop study succeed suffer suggest support suppose surprise survive swim take
    talk teach tell tend test thank think throw touch train travel treat try turn understand use visit vote
    wait walk want warn wash watch wear win wish wonder work worry write jog chaperone premiere locate situate
    organize operate publish compose conduct direct produce record release star feature wish pursue further
    furthermore teach lecture tour hike fish camp skate ski surf bowl paint draw sketch photograph apply
    register enroll attend graduate retire resign hire fire employ dislike admire respect trust fear annoy
    bother amuse entertain inspire influence impress defend attack assist advise recommend suggest request
    demand declare state report examine inspect investigate analyze measure weigh calculate predict estimate
    join leave arrive depart enter exit cross pass reach approach visit host invite welcome greet meet
    contribute donate give lend borrow rent lease buy sell trade exchange sponsor fund finance own possess
    """.split()


    def stem(word: str) -> str:
        w = word.lower()
        if w in IRREGULAR_STEMS:
            return IRREGULAR_STEMS[w]
        if w in IRREGULAR_PAST and w not in {"left", "saw", "found", "set", "put", "bit"}:
            return IRREGULAR_PAST[w]
        if "'" in w or "’" in w or any(c.isdigit() for c in w) or len(w) <= 3:
            return w
        if w.endswith("ies") and len(w) > 4:
            w = w[:-3] + "y"
        elif w.endswith("sses"):
            w = w[:-2]
        elif re.search(r"(ches|shes|xes|zes)$", w):
            w = w[:-2]
        elif w.endswith("ing") and len(w) > 5:
            w = w[:-3]
        elif w.endswith("ed") and len(w) > 4:
            w = w[:-2]
        elif w.endswith("s") and not w.endswith(("ss", "us", "is")):
            w = w[:-1]
        if w.endswith("e") and len(w) > 3:
            w = w[:-1]
        return w


    VERB_STEMS = {stem(v) for v in BASE_VERBS}


    def is_verb(word: str) -> bool:
        w = word.lower()
        if w in DETERMINERS or w in PREPOSITIONS:
            return False
        return w in IRREGULAR_PAST or stem(w) in VERB_STEMS


    # --- parsing/nl_logic.py ---
    WORD = re.compile(r"[A-Za-z0-9$%][A-Za-z0-9$%'’\-\.]*[A-Za-z0-9$%]|[A-Za-z0-9$%]|[,;]")
    CONTRACTIONS = {"don": "do", "doesn": "does", "didn": "did", "isn": "is", "aren": "are", "wasn": "was",
                    "weren": "were", "hasn": "has", "haven": "have", "hadn": "had", "won": "will",
                    "can": "can", "couldn": "could", "wouldn": "would", "shouldn": "should"}
    NAME_CONNECTORS = {"of", "de", "von", "van", "der", "la", "le", "du", "del", "the"}
    RELATIVE = {"who", "that", "which", "whom"}
    PLURAL_VERBS = {"are", "were", "do", "have"}
    BAD_NP_END = RELATIVE | {"and", "or", ",", "of", "in", "on", "at", "by", "with", "to", "for", "from",
                             "the", "a", "an", "not", "either", "neither", "both", "nor", ";"}
    CLAUSE_STARTERS = {"he", "she", "they", "it", "there", "all", "every", "each", "no", "some", "any",
                       "someone", "everyone", "nobody", "something", "everything", "if", "either", "neither"}


    @dataclass(frozen=True, slots=True)
    class Tok:
        text: str
        low: str
        cap: bool


    def tokenize(sentence: str) -> list[Tok]:
        toks: list[Tok] = []
        sentence = re.sub(r"\[[^\]]*\]|\([^)]*\)", " ", sentence)
        sentence = re.sub(r"[“\"]([^”\"]+)[”\"]", lambda m: " " + re.sub(r"[^A-Za-z0-9]+", "", m.group(1)).capitalize() + " ", sentence)
        sentence = re.sub(r",?\s+or both\b|\band/or\b", " or", sentence)
        for m in WORD.finditer(sentence):
            w = m.group(0)
            low = w.lower().replace("’", "'")
            if low == "cannot":
                toks += [Tok("can", "can", False), Tok("not", "not", False)]
            elif low.endswith("n't") and len(low) > 3:
                base = CONTRACTIONS.get(low[:-3], low[:-3])
                toks += [Tok(base, base, w[0].isupper()), Tok("not", "not", False)]
            else:
                toks.append(Tok(w, low, w[0].isupper() or w[0].isdigit()))
        return [t for t in toks if t.low not in FILLERS]


    def is_possessive(tok: Tok) -> bool:
        return tok.low.endswith("'s") or tok.low.endswith("s'")


    def split_top(toks: list[Tok], separators: set[str]) -> list[list[Tok]]:
        parts: list[list[Tok]] = [[]]
        for t in toks:
            if t.low in separators:
                parts.append([])
            else:
                parts[-1].append(t)
        return parts


    def strip_commas(toks: list[Tok]) -> list[Tok]:
        start, end = 0, len(toks)
        while start < end and toks[start].low in {",", ";"}:
            start += 1
        while end > start and toks[end - 1].low in {",", ";"}:
            end -= 1
        return toks[start:end]


    def negate(f: Formula) -> Formula:
        return f.f if isinstance(f, Not) else Not(f)


    Builder = Callable[[Term], Formula]


    class SentenceParser:
        def __init__(self, exclusive_either: bool = True):
            self.exclusive_either = exclusive_either
            self.var: Var | None = None
            self.var_noun: str | None = None
            self.quant = "forall"
            self.restrict: list[Formula] = []
            self.name: str | None = None
            self.antecedent = False

        def parse(self, text: str) -> Formula:
            toks = strip_commas(tokenize(text))
            if not toks:
                raise ParseError(text)
            return self.close(self.formula(toks))

        def state(self):
            return (self.var, self.var_noun, self.quant, list(self.restrict), self.name, self.antecedent)

        def restore(self, saved) -> None:
            self.var, self.var_noun, self.quant, restrict, self.name, self.antecedent = saved
            self.restrict = list(restrict)

        def close(self, f: Formula) -> Formula:
            if self.var is None:
                return f
            restrictors = tuple(self.restrict)
            if self.quant == "exists":
                return Exists(self.var, And(restrictors + (f,)))
            return Forall(self.var, Imp(And(restrictors), f) if restrictors else f)

        def formula(self, toks: list[Tok]) -> Formula:
            toks = strip_commas(toks)
            if not toks:
                raise ParseError("empty")
            words = [t.low for t in toks]
            if words[:3] == ["it", "is", "not"] and len(words) > 4 and words[3] in {"the", "true", "possible"}:
                if "that" in words:
                    return Not(self.formula(toks[words.index("that") + 1:]))
            if words[0] in {"if", "when", "whenever"}:
                if words[:4] == ["if", "and", "only", "if"]:
                    return self.conditional(toks[3:])
                return self.conditional(toks)
            for i in range(1, len(toks) - 3):
                if words[i:i + 4] == ["if", "and", "only", "if"]:
                    left = self.formula(toks[:i])
                    return Iff(left, self.formula(toks[i + 4:]))
            for i in range(1, len(toks)):
                if words[i] in {"if", "unless", "when", "whenever"} and i + 1 < len(toks):
                    left, right = toks[:i], toks[i + 1:]
                    if words[i] == "unless":
                        return Or((self.formula(left), self.formula(right)))
                    only = left and left[-1].low == "only"
                    consequent = self.formula(left[:-1] if only else left)
                    antecedent = self.formula(right)
                    return Imp(consequent, antecedent) if only else Imp(antecedent, consequent)
            split = self.split_clauses(toks)
            if split is not None:
                return split
            return self.clause(toks)

        def conditional(self, toks: list[Tok]) -> Formula:
            words = [t.low for t in toks]
            if "then" in words:
                t = words.index("then")
                ant, cons = toks[1:t], toks[t + 1:]
            else:
                if "," not in words:
                    raise ParseError("if without then")
                c = words.index(",")
                ant, cons = toks[1:c], toks[c + 1:]
            self.antecedent = True
            a = self.formula(ant)
            self.antecedent = False
            return Imp(a, self.formula(cons))

        def starts_clause(self, toks: list[Tok], i: int) -> bool:
            if i >= len(toks):
                return False
            t = toks[i]
            rest = [x.low for x in toks[i:]]
            has_verb = any(w in COPULAS or w in DO or w in HAVE or w in MODALS or is_verb(w) for w in rest[1:])
            if t.low in CLAUSE_STARTERS:
                return True
            if t.low in DETERMINERS:
                return any(w in COPULAS or w in DO or w in HAVE for w in rest)
            if t.cap and not is_possessive(t):
                return has_verb
            return False

        def split_clauses(self, toks: list[Tok]) -> Formula | None:
            n = len(toks)
            for separators, combine in (({"or"}, "or"), ({"but", ";"}, "and"), ({"and"}, "and")):
                for i in range(1, n - 1):
                    if toks[i].low not in separators:
                        continue
                    left = strip_commas(toks[:i])
                    right_start = i + 1
                    if not self.starts_clause(toks, right_start):
                        continue
                    right = toks[right_start:]
                    exclusive = False
                    if combine == "or":
                        either = [k for k, t in enumerate(left) if t.low == "either"]
                        if either:
                            left = left[:either[0]] + left[either[0] + 1:]
                            exclusive = self.exclusive_either
                        elif left and left[0].low == "neither":
                            continue
                    saved = self.state()
                    try:
                        lf = self.formula(left)
                        rf = self.formula(right)
                    except ParseError:
                        self.restore(saved)
                        continue
                    if combine == "or":
                        return Xor(lf, rf) if exclusive else Or((lf, rf))
                    return And((lf, rf))
            return None

        def pronoun(self) -> Term:
            if self.var is not None:
                return self.var
            if self.name is not None:
                return self.name
            raise ParseError("unresolved pronoun")

        def name_run(self, toks: list[Tok], i: int) -> int:
            j = i
            while j < len(toks):
                t = toks[j]
                if (t.cap or t.text[0].isdigit()) and not is_possessive(t) and t.low not in {"if", "then"}:
                    j += 1
                elif t.low in NAME_CONNECTORS and j > i and j + 1 < len(toks) and toks[j + 1].cap:
                    j += 1
                else:
                    break
            return j

        def name_term(self, toks: list[Tok]) -> str:
            words = [t.low for t in toks]
            if words and words[0] == "the":
                words = words[1:]
            return " ".join(words)

        def norm(self, toks: list[Tok]) -> str | None:
            words = [stem(t.low) for t in toks if t.low not in DETERMINERS and t.low not in {",", ";"}]
            words = [w for w in words if w not in GENERIC_NOUNS and w != "person"] or []
            return " ".join(words) if words else None

        def vp_start(self, toks: list[Tok], j: int) -> bool:
            low = toks[j].low
            if j > 0 and toks[j - 1].low in DETERMINERS | PREPOSITIONS | {"and", "or", ",", "not"} | RELATIVE:
                return False
            return (low in COPULAS or low in DO or low in HAVE or low in MODALS
                    or low in {"either", "both", "neither", "never"} or is_verb(low))

        def with_subject(self, toks: list[Tok], i0: int, build: Callable[[list[Tok]], Term],
                         negated: bool = False, allow_empty: bool = False) -> Formula:
            for j in range(i0 if allow_empty else i0 + 1, len(toks)):
                if not self.vp_start(toks, j):
                    continue
                np = strip_commas(toks[i0:j])
                if not np and not allow_empty:
                    continue
                if np and np[-1].low in BAD_NP_END:
                    continue
                saved = self.state()
                try:
                    subject = build(np)
                    body = self.vp(toks[j:])(subject)
                    return negate(body) if negated else body
                except ParseError:
                    self.restore(saved)
            raise ParseError("no subject/verb split")

        def start_generic(self, np: list[Tok], quant: str) -> Var:
            if self.var is not None:
                raise ParseError("two generic subjects")
            var = Var("x")
            self.var = var
            self.quant = "forall" if (quant in {"forall", "none"} or self.antecedent) else "exists"
            self.restrict = []
            r = next((k for k, t in enumerate(np) if t.low in RELATIVE), None)
            head = np if r is None else np[:r]
            self.var_noun = self.norm(head)
            pieces = []
            if head:
                f = self.phrase_formula(var, head)
                if f is not None:
                    pieces.append(f)
            if r is not None:
                pieces.append(self.vp(np[r + 1:])(var))
            self.restrict = pieces
            return var

        def clause(self, toks: list[Tok]) -> Formula:
            first = toks[0]
            low = first.low
            if low == "there":
                return self.existential(toks)
            if low in PRONOUNS:
                return self.vp(toks[1:])(self.pronoun())
            if low in GENERIC_PRONOUN_QUANTIFIERS:
                q = GENERIC_PRONOUN_QUANTIFIERS[low]
                return self.with_subject(toks, 1, lambda np: self.start_generic(np, q), negated=(q == "none"),
                                         allow_empty=True)
            if low in QUANTIFIERS:
                q = QUANTIFIERS[low]
                i = 1
                if i < len(toks) and toks[i].low == "of":
                    i += 1
                if i < len(toks) and toks[i].low == "the":
                    i += 1
                if low == "no" and i < len(toks) and toks[i].low == "one":
                    return self.with_subject(toks, i + 1, lambda np: self.start_generic(np, q), negated=True,
                                             allow_empty=True)
                return self.with_subject(toks, i, lambda np: self.start_generic(np, q), negated=(q == "none"))
            if low in {"a", "an"}:
                return self.with_subject(toks, 1, lambda np: self.start_generic(np, "forall"))
            if low == "the":
                return self.with_subject(toks, 1, self.definite)
            j = self.name_run(toks, 0)
            if j > 0:
                nxt = toks[j].low if j < len(toks) else None
                plural = nxt in PLURAL_VERBS or (j == 1 and nxt is not None and is_verb(nxt)
                                                  and not nxt.endswith(("s", "ed")) and nxt not in COPULAS
                                                  and low.endswith("s") and low not in {"is", "was"})
                if nxt == "and":
                    k = self.name_run(toks, j + 1)
                    if k > j + 1:
                        names = [self.name_term(toks[:j]), self.name_term(toks[j + 1:k])]
                        parts = [self.vp(toks[k:])(n) for n in names]
                        return And(tuple(parts))
                if not plural and j < len(toks) and low not in GENERIC_NOUNS and stem(low) not in GENERIC_NOUNS:
                    saved = self.state()
                    subject = self.name_term(toks[:j])
                    try:
                        self.name = subject
                        return self.vp(toks[j:])(subject)
                    except ParseError:
                        self.restore(saved)
            return self.with_subject(toks, 0, lambda np: self.start_generic(np, "forall"))

        def definite(self, np: list[Tok]) -> Term:
            if not np:
                raise ParseError("empty definite")
            if all(t.cap or t.low in NAME_CONNECTORS for t in np) and not is_possessive(np[0]):
                term = self.name_term(np)
            else:
                if any(t.low in RELATIVE or t.low == "," for t in np):
                    raise ParseError("complex definite")
                noun = self.norm(np)
                if noun is None:
                    raise ParseError("generic definite")
                if self.var is not None and self.var_noun == noun:
                    return self.var
                term = noun
            self.name = term
            return term

        def existential(self, toks: list[Tok]) -> Formula:
            words = [t.low for t in toks]
            if len(toks) < 3 or words[1] not in COPULAS | {"exists", "exist"}:
                raise ParseError("there")
            i = 2
            negated = False
            if i < len(toks) and toks[i].low in {"no"}:
                negated = True
                i += 1
            elif i < len(toks) and toks[i].low in {"a", "an", "some", "one", "at", "several"}:
                i += 1
                if toks[i - 1].low == "at" and i < len(toks) and toks[i].low == "least":
                    i += 2
            np = toks[i:]
            if not np:
                raise ParseError("empty existential")
            var = Var("x")
            r = next((k for k, t in enumerate(np) if t.low in RELATIVE), None)
            head = np if r is None else np[:r]
            pieces = []
            f = self.phrase_formula(var, head)
            if f is not None:
                pieces.append(f)
            if r is not None:
                pieces.append(self.vp(np[r + 1:])(var))
            body = Exists(var, And(tuple(pieces)))
            return Not(body) if negated else body

        def vp(self, toks: list[Tok]) -> Builder:
            toks = strip_commas(toks)
            if not toks:
                raise ParseError("empty vp")
            low = toks[0].low
            if low == "either":
                parts = [self.vp(p) for p in split_top(toks[1:], {"or"}) if p]
                if len(parts) < 2:
                    raise ParseError("either")
                return lambda s: self.combine_or([p(s) for p in parts], self.exclusive_either)
            if low == "neither":
                parts = [self.vp(p) for p in split_top(toks[1:], {"nor"}) if p]
                if len(parts) < 2:
                    raise ParseError("neither")
                return lambda s: And(tuple(Not(p(s)) for p in parts))
            if low == "both":
                parts = [self.vp(p) for p in split_top(toks[1:], {"and"}) if p]
                return lambda s: And(tuple(p(s) for p in parts))
            groups = self.vp_coordination(toks)
            if groups is not None:
                builders, ops = groups
                return lambda s: self.fold([b(s) for b in builders], ops)
            return self.simple_vp(toks)

        def combine_or(self, parts: list[Formula], exclusive: bool) -> Formula:
            if len(parts) == 2 and exclusive:
                return Xor(parts[0], parts[1])
            return Or(tuple(parts))

        def fold(self, parts: list[Formula], ops: list[str]) -> Formula:
            groups = [[parts[0]]]
            for p, op in zip(parts[1:], ops):
                if op == "or":
                    groups.append([p])
                else:
                    groups[-1].append(p)
            ands = [g[0] if len(g) == 1 else And(tuple(g)) for g in groups]
            return ands[0] if len(ands) == 1 else Or(tuple(ands))

        def vp_coordination(self, toks: list[Tok]) -> tuple[list[Builder], list[str]] | None:
            cuts: list[tuple[int, str, int]] = []
            pending = {"and": 0, "or": 0}
            for i in range(1, len(toks)):
                if toks[i].low == "both":
                    pending["and"] += 1
                elif toks[i].low == "either":
                    pending["or"] += 1
                elif toks[i].low in pending and pending[toks[i].low]:
                    pending[toks[i].low] -= 1
                    continue
                if toks[i].low in {"and", "or", "but", ","}:
                    k = i + 1
                    op = {"and": "and", "or": "or", "but": "and", ",": "and"}[toks[i].low]
                    while k < len(toks) and toks[k].low in {"and", "or", "but", ","}:
                        if toks[k].low == "or":
                            op = "or"
                        k += 1
                    if k < len(toks) and self.vp_start_after_cut(toks, k):
                        cuts.append((i, op, k))
            if not cuts:
                return None
            builders, ops, start = [], [], 0
            for i, op, k in cuts:
                builders.append(self.vp(toks[start:i]))
                ops.append(op)
                start = k
            builders.append(self.vp(toks[start:]))
            return builders, ops

        def vp_start_after_cut(self, toks: list[Tok], k: int) -> bool:
            low = toks[k].low
            return (low in COPULAS or low in DO or low in HAVE or low in MODALS
                    or low in {"either", "neither", "both", "never"} or is_verb(low))

        def simple_vp(self, toks: list[Tok]) -> Builder:
            n = len(toks)
            i = 0
            neg = False
            while i < n and (toks[i].low in DO or toks[i].low in MODALS):
                i += 1
                if i < n and toks[i].low == "not":
                    neg = True
                    i += 1
            if i < n and toks[i].low == "never":
                neg = True
                i += 1
            if i >= n:
                raise ParseError("empty verb phrase")
            low = toks[i].low
            if low in COPULAS:
                i += 1
                if i < n and toks[i].low == "not":
                    neg = not neg
                    i += 1
                comp = self.complement(toks[i:])
                return (lambda s: negate(comp(s))) if neg else comp
            if low in HAVE:
                i += 1
                if i < n and toks[i].low == "not":
                    neg = True
                    i += 1
                rest = [Tok("have", "have", False)] + toks[i:]
            else:
                if low in DETERMINERS or low in PREPOSITIONS or low in {"and", "or", "if", "then", "that", "who", "which"}:
                    raise ParseError("not a verb phrase")
                rest = toks[i:]
            if any(t.low in COPULAS or t.low in DO or t.low in MODALS for t in rest):
                raise ParseError("stray auxiliary")

            def build(s: Term) -> Formula:
                f = self.phrase_formula(s, rest)
                if f is None:
                    raise ParseError("empty predicate")
                return negate(f) if neg else f

            return build

        def complement(self, toks: list[Tok]) -> Builder:
            toks = strip_commas(toks)
            if not toks:
                raise ParseError("empty complement")
            low = toks[0].low
            if low == "either":
                parts = [self.complement(p) for p in split_top(toks[1:], {"or"}) if p]
                return lambda s: self.combine_or([p(s) for p in parts], self.exclusive_either)
            if low == "neither":
                parts = [self.complement(p) for p in split_top(toks[1:], {"nor"}) if p]
                return lambda s: And(tuple(Not(p(s)) for p in parts))
            if low == "both":
                parts = [self.complement(p) for p in split_top(toks[1:], {"and"}) if p]
                return lambda s: And(tuple(p(s) for p in parts))
            r = next((k for k, t in enumerate(toks) if t.low in RELATIVE), len(toks))
            for sep, joiner in (({"or"}, "or"), ({"and"}, "and")):
                cut = next((k for k in range(1, r) if toks[k].low in sep), None)
                if cut is not None:
                    left, right = self.complement(toks[:cut]), self.complement(toks[cut + 1:])
                    if joiner == "or":
                        return lambda s: Or((left(s), right(s)))
                    return lambda s: And((left(s), right(s)))
            return self.np_complement(toks)

        def np_complement(self, toks: list[Tok]) -> Builder:
            r = next((k for k, t in enumerate(toks) if t.low in RELATIVE), None)
            if r is None:
                def build(s: Term) -> Formula:
                    f = self.phrase_formula(s, toks)
                    if f is None:
                        raise ParseError("empty complement")
                    return f
                return build
            head, rel = toks[:r], self.vp(toks[r + 1:])

            def build_rel(s: Term) -> Formula:
                pieces = []
                f = self.phrase_formula(s, head) if head else None
                if f is not None:
                    pieces.append(f)
                pieces.append(rel(s))
                return pieces[0] if len(pieces) == 1 else And(tuple(pieces))

            return build_rel

        def is_name_start(self, toks: list[Tok], k: int) -> bool:
            t = toks[k]
            if not (t.cap or t.text[0].isdigit()) or is_possessive(t):
                return False
            b = k - 1
            if b >= 0 and toks[b].low == "the":
                b -= 1
            if b < 0:
                return False
            return toks[b].low in PREPOSITIONS or is_verb(toks[b].low)

        def phrase_formula(self, subject: Term, toks: list[Tok]) -> Formula | None:
            toks = strip_commas(toks)
            spans: list[tuple[int, int]] = []
            k = 0
            while k < len(toks):
                if self.is_name_start(toks, k):
                    end = self.name_run(toks, k)
                    if end == len(toks):
                        spans.append((k, end))
                    k = end
                else:
                    k += 1
            if not spans:
                name = self.norm(toks)
                return None if name is None else Pred(name, (subject,))
            if len(spans) > 1:
                raise ParseError("several names")
            s, e = spans[0]
            if e < len(toks):
                raise ParseError("material after name")
            left = toks[:s]
            while left and left[-1].low == "the":
                left = left[:-1]
            if not left:
                raise ParseError("bare name")
            term = self.name_term(toks[s:e])
            last = left[-1]
            if last.low in PREPOSITIONS:
                before = left[:-1]
                if not before:
                    return Pred(last.low, (subject, term))
                if is_verb(before[-1].low):
                    return Pred(self.norm(left) or last.low, (subject, term))
                head = self.norm(before)
                relation = Pred(last.low, (subject, term))
                return relation if head is None else And((Pred(head, (subject,)), relation))
            name = self.norm(left)
            if name is None:
                raise ParseError("empty relation")
            return Pred(name, (subject, term))


    def parse_formula(text: str, exclusive_either: bool = True) -> Formula:
        return SentenceParser(exclusive_either).parse(text)

    class ParseError(ValueError):
        pass


    def fol_open_world_entails(premises, conclusion, timeout=1.0):
        """English -> first-order formulas (rule-based) -> CNF -> resolution. Unparsable premises are skipped."""
        lines = premises if isinstance(premises, list) else str(premises).split("\n")
        formulas = []
        for line in lines:
            if line and line.strip():
                try:
                    formulas.append(parse_formula(line))
                except ParseError:
                    continue
        try:
            goal = parse_formula(conclusion)
        except ParseError:
            return False
        return prove(premise_clauses(formulas), goal, timeout=timeout).proved


    return fol_open_world_entails


fol_open_world_entails = _build_fol_engine()



In [7]:
DOWNLOAD_DATA = False
if DOWNLOAD_DATA:
    from huggingface_hub import login
    login()


In [8]:
if DOWNLOAD_DATA:
    JOBS = [
        ("hitachi-nlp/ruletaker", None, "train", "ruletaker_train.jsonl"),
        ("hitachi-nlp/ruletaker", None, "test", "ruletaker_test.jsonl"),
        ("yale-nlp/FOLIO", None, "train", "FOLIO_train.jsonl"),
        ("yale-nlp/FOLIO", None, "validation", "FOLIO_validation.jsonl"),

        # RuleTaker out-of-distribution test set
        ("jise/ruletaker", None, "ood_test", "ruletaker_ood_test.jsonl"),

        # CounterBench Alpha: download and convert the JSON file separately
        (None, None, None, "counterbench_alpha.jsonl"),
    ]


In [9]:
if DOWNLOAD_DATA:
    import json
    from pathlib import Path
    from urllib.request import urlopen

    data_dir = Path("Data")
    data_dir.mkdir(exist_ok=True)

    url = (
        "https://huggingface.co/datasets/CounterBench/CounterBench/"
        "resolve/main/data_balanced_alpha_V1.json"
    )

    json_path = data_dir / "data_balanced_alpha_V1.json"
    jsonl_path = data_dir / "counterbench_alpha.jsonl"

    if not jsonl_path.exists():
        with urlopen(url, timeout=60) as response:
            raw_data = json.load(response)

        # Handle either a list of records or a dictionary containing records.
        if isinstance(raw_data, dict):
            for key in ("data", "records", "examples"):
                if isinstance(raw_data.get(key), list):
                    raw_data = raw_data[key]
                    break

        if not isinstance(raw_data, list):
            raise ValueError(
                "Unexpected CounterBench JSON structure. "
                "Inspect the downloaded JSON before converting it."
            )

        with jsonl_path.open("w", encoding="utf-8") as f:
            for row in raw_data:
                f.write(json.dumps(row, ensure_ascii=False) + "\n")

        print(f"Saved {len(raw_data)} records to {jsonl_path}")
    else:
        print("Already exists:", jsonl_path)


In [10]:
if DOWNLOAD_DATA:
    from datasets import load_dataset
    import json
    from pathlib import Path

    data_dir = Path("Data")
    output_file = data_dir / "ruletaker_ood_test.jsonl"

    try:
        dataset = load_dataset(
            "jise/ruletaker",
            split="ood_test"
        )

        with output_file.open("w", encoding="utf-8") as f:
            for record in dataset:
                f.write(json.dumps(record, ensure_ascii=False) + "\n")

        print(f"Downloaded {len(dataset)} records to:")
        print(output_file)

    except Exception as e:
        print("Download failed:", type(e).__name__, str(e))


In [11]:
if DOWNLOAD_DATA:
    # Downloads the dataset files into ./Data (skips files already present)
    import json
    from pathlib import Path
    from datasets import load_dataset

    DATA = Path("Data")
    DATA.mkdir(exist_ok=True)

    # (Hugging Face repo, config or None, split, output file)
    JOBS = [
        ("hitachi-nlp/ruletaker", None, "train", "ruletaker_train.jsonl"),
        ("hitachi-nlp/ruletaker", None, "test", "ruletaker_test.jsonl"),
        ("yale-nlp/FOLIO", None, "train", "FOLIO_train.jsonl"),
        ("yale-nlp/FOLIO", None, "validation", "FOLIO_validation.jsonl"),
        # Fill these in from the source your course gives you. Skipped until you do.
        (None, None, None, "ruletaker_ood_test.jsonl"),
        (None, None, None, "counterbench_alpha.jsonl"),
    ]

    for repo, config, split, out_name in JOBS:
        out_path = DATA / out_name
        if out_path.exists():
            print("already there:", out_name)
            continue
        if repo is None:
            print("SKIPPED (no source set):", out_name)
            continue
        try:
            ds = load_dataset(repo, config, split=split)
            with open(out_path, "w", encoding="utf-8") as f:
                for row in ds:
                    f.write(json.dumps(row, ensure_ascii=False) + "\n")
            print(f"saved {out_name}: {len(ds)} rows")
        except Exception as e:
            print(f"FAILED {out_name}: {type(e).__name__}: {str(e)[:200]}")


## 6. Pipeline and Evaluation

`evaluate_record(record, fallback_key)` selects the engine from the input fields:

- `given_info`, `question`: CounterBench
- `premises`, `conclusion`: FOLIO (NL fallback if needed)
- `context`, `statement`: RuleTaker

Returns `(id, predicted_bool, gold_bool_or_None)`. The final cell runs the example, evaluates the datasets and writes `submission.csv`.


In [12]:
# =====================================================================
# DATASET LOCATOR: finds the .jsonl files and copies them into ./Data
# Run this cell BEFORE the evaluation cell.
# =====================================================================
import os
import shutil
from pathlib import Path

REQUIRED_DATASETS = [
    "ruletaker_train.jsonl",
    "ruletaker_test.jsonl",
    "ruletaker_ood_test.jsonl",
    "counterbench_alpha.jsonl",
    "FOLIO_train.jsonl",
    "FOLIO_validation.jsonl",
]

NOTEBOOK_DIR = Path.cwd()
DATA_DIR = NOTEBOOK_DIR / "Data"
SEARCH_DIRS = [
    DATA_DIR,
    NOTEBOOK_DIR,
    Path.home() / "Downloads",
    Path.home() / "Desktop",
    Path.home() / "Documents",
]


def find_dataset(name):
    """Return the path of the first copy of `name` found, or None."""
    # 1. Quick check directly inside the usual folders
    for folder in SEARCH_DIRS:
        if (folder / name).is_file():
            return folder / name
    # 2. Slower check inside subfolders as well
    for folder in SEARCH_DIRS:
        if not folder.is_dir():
            continue
        try:
            for dirpath, _, filenames in os.walk(folder):
                if name in filenames:
                    return Path(dirpath) / name
        except OSError:
            continue
    return None


DATA_DIR.mkdir(exist_ok=True)
located = {}
for name in REQUIRED_DATASETS:
    if (DATA_DIR / name).is_file():
        located[name] = DATA_DIR / name
        continue
    source = find_dataset(name)
    if source is not None:
        shutil.copy2(source, DATA_DIR / name)
        located[name] = source

print(f"Notebook folder: {NOTEBOOK_DIR}")
for name in REQUIRED_DATASETS:
    status = "FOUND  " if name in located else "MISSING"
    print(f"{status} {name}" + (f"   (from {located[name]})" if name in located else ""))

missing = [n for n in REQUIRED_DATASETS if n not in located]
if missing:
    print(f"\n{len(missing)} file(s) not found anywhere. Copy them into this folder: {DATA_DIR}")
else:
    print(f"\nAll {len(REQUIRED_DATASETS)} dataset files are in: {DATA_DIR}")

Notebook folder: e:\RegularAI\AI_Assignment
FOUND   ruletaker_train.jsonl   (from e:\RegularAI\AI_Assignment\Data\ruletaker_train.jsonl)
FOUND   ruletaker_test.jsonl   (from e:\RegularAI\AI_Assignment\Data\ruletaker_test.jsonl)
FOUND   ruletaker_ood_test.jsonl   (from e:\RegularAI\AI_Assignment\Data\ruletaker_ood_test.jsonl)
FOUND   counterbench_alpha.jsonl   (from e:\RegularAI\AI_Assignment\Data\counterbench_alpha.jsonl)
FOUND   FOLIO_train.jsonl   (from e:\RegularAI\AI_Assignment\Data\FOLIO_train.jsonl)
FOUND   FOLIO_validation.jsonl   (from e:\RegularAI\AI_Assignment\Data\FOLIO_validation.jsonl)

All 6 dataset files are in: e:\RegularAI\AI_Assignment\Data


In [13]:

# =====================================================================
# 5b. RULETAKER REASONER
# Unary and binary Horn rules with variables, negated facts and
# conditions, universal rules, CWA and OWA semantics.
# =====================================================================

import re
from collections import defaultdict
from itertools import product
from functools import lru_cache


RT_GENERIC = {
    "something", "someone", "somebody", "anything", "anyone",
    "everything", "everyone", "nothing", "nobody", "thing",
    "person", "things", "people"
}

RT_PRONOUNS = {"it", "they", "he", "she"}
RT_ARTICLES = {"the", "a", "an"}
RT_NOUN_TAILS = ("people", "things", "thing", "person", "persons")


def rt_words(text):
    """Lower-case word tokens with articles removed."""
    text = str(text).strip().rstrip(".!?").strip().lower()
    return [
        w for w in re.findall(r"[a-z]+", text)
        if w not in RT_ARTICLES
    ]


def rt_singular(word):
    if len(word) > 3 and word.endswith("s") and not word.endswith("ss"):
        return word[:-1]
    return word


def rt_norm_verb(word):
    """'likes' and 'like' map to the same predicate name."""
    if word.endswith(("ches", "shes", "sses", "xes", "oes")):
        return word[:-2]
    if word.endswith("s") and not word.endswith("ss"):
        return word[:-1]
    return word


def rt_subject(word):
    return "?x" if (
        word in RT_GENERIC or word in RT_PRONOUNS
    ) else word


def rt_object(words):
    key = " ".join(words)
    return "?y" if (
        key in RT_GENERIC or key in RT_PRONOUNS
    ) else key


def rt_clause(text):
    """One simple sentence -> list of (predicate, args, negated)."""
    w = rt_words(text)

    if len(w) < 2:
        return []

    subj = rt_subject(w[0])

    if w[1] in ("is", "are"):
        rest = w[2:]
        neg = False

        if rest and rest[0] == "not":
            neg, rest = True, rest[1:]

        if not rest:
            return []

        return [(rt_singular(" ".join(rest)), (subj,), neg)]

    if len(w) >= 5 and w[1] in ("does", "do") and w[2] == "not":
        return [
            (rt_norm_verb(w[3]), (subj, rt_object(w[4:])), True)
        ]

    if len(w) >= 3:
        return [
            (rt_norm_verb(w[1]), (subj, rt_object(w[2:])), False)
        ]

    return [(rt_norm_verb(w[1]), (subj,), False)]


def rt_literals(text):
    """A conjunction -> list of literals."""
    parts = [
        p.strip()
        for p in re.split(r"\s*,\s*|\s+and\s+", text)
        if p.strip()
    ]

    lits, subj = [], None

    for part in parts:
        ws = rt_words(part)

        if not ws:
            continue

        if ws[0] == "not" or len(ws) == 1:
            if subj is None:
                continue

            part = f"{subj} is {part}"
            ws = rt_words(part)

        clause = rt_clause(part)

        if clause:
            lits.extend(clause)
            subj = ws[0]

    return lits


def rt_universal(low):
    """Parse universal rules such as 'All young people are smart'."""
    m = re.match(
        r"^(all\s+)?(.+?)\s+(?:are|is)\s+(.+)$",
        low
    )

    if not m:
        return None

    is_all, subj, head = m.group(1), m.group(2), m.group(3)

    if not is_all and not re.search(
        r"\b(" + "|".join(RT_NOUN_TAILS) + r")$",
        subj.strip()
    ):
        return None

    body = [
        rt_singular(w)
        for w in re.split(r"[\s,]+", subj)
        if w and w not in ("all", "and", "the")
        and w not in RT_NOUN_TAILS
    ]

    if not body:
        return None

    conds = [(b, ("?x",), False) for b in body]
    concls = []

    for part in re.split(r"\s*,\s*|\s+and\s+", head):
        ws = part.split()

        if not ws:
            continue

        neg = ws[0] == "not"

        if neg:
            ws = ws[1:]

        if ws:
            concls.append(
                (rt_singular(" ".join(ws)), ("?x",), neg)
            )

    return (conds, concls) if concls else None



def rt_parse(context):
    """Context text -> (set of ground facts, list of rules)."""
    facts, rules = set(), []

    # Recognize the different RuleTaker rule-writing formats.
    rule_patterns = [
        r"^if\s+(.+?)\s*,?\s+then\s+(.+)$",
        r"^based on\s+(.+?),\s*it is evident that\s+(.+)$",
        r"^assuming\s+(.+?),\s*we can say\s+(.+)$",
        r"^when\s+(.+?),\s*we can conclude\s+(.+)$",
        r"^given that\s+(.+?),\s*it follows that\s+(.+)$"
    ]

    for raw in re.split(
        r"(?<=[.!?])\s+|\n+",
        str(context or "")
    ):
        s = raw.strip().rstrip(".!?").strip()

        s = re.sub(
            r"^(facts|rules)\s*:\s*",
            "",
            s,
            flags=re.IGNORECASE
        ).strip()

        if not s or s.lower() in ("facts", "rules"):
            continue

        low = s.lower()

        # Try all supported rule formats.
        matched_rule = False

        for pattern in rule_patterns:
            m = re.match(pattern, low)

            if not m:
                continue

            conds = rt_literals(m.group(1))
            concls = rt_literals(m.group(2))

            if not conds or not concls:
                matched_rule = True
                break

            # Keep the existing variable-safety check.
            cvars = {
                v
                for _, args, _ in conds
                for v in args
                if v.startswith("?")
            }

            concls = [
                c for c in concls
                if {
                    v for v in c[1] if v.startswith("?")
                } <= cvars
            ]

            if concls:
                rules.append((conds, concls))

            matched_rule = True
            break

        if matched_rule:
            continue

        # Preserve existing universal-rule parsing.
        uni = rt_universal(low)

        if uni:
            rules.append(uni)
            continue

        # Otherwise, treat it as a fact.
        for lit in rt_literals(s):
            if not any(a.startswith("?") for a in lit[1]):
                facts.add(lit)

    return facts, rules


def _rt_unify(args, fargs, b):
    if len(args) != len(fargs):
        return None

    nb = dict(b)

    for a, f in zip(args, fargs):
        if a.startswith("?"):
            if a in nb:
                if nb[a] != f:
                    return None
            else:
                nb[a] = f

        elif a != f:
            return None

    return nb


class RuleTakerReasoner:
    """Forward chaining with CWA or OWA negation."""

    def __init__(self, facts, rules, query_args=(), open_world=False):
        self.facts = facts
        self.rules = rules
        self.open_world = open_world

        consts = set()

        for _, args, _ in facts:
            consts.update(args)

        for conds, concls in rules:
            for _, args, _ in conds + concls:
                consts.update(
                    a for a in args if not a.startswith("?")
                )

        consts.update(query_args)
        self.domain = sorted(consts)

    def _neg_holds(self, cond, b, pos, neg):
        key = (
            cond[0],
            tuple(b.get(a, a) for a in cond[1])
        )

        return (
            key in neg
            if self.open_world
            else key not in pos
        )

    def _bindings(self, conds, index, pos, neg):
        pos_conds = [c for c in conds if not c[2]]
        neg_conds = [c for c in conds if c[2]]

        out = []

        def extend(i, b):
            if i == len(pos_conds):
                free = []

                for _, args, _ in neg_conds:
                    for v in args:
                        if (
                            v.startswith("?")
                            and v not in b
                            and v not in free
                        ):
                            free.append(v)

                for combo in product(
                    self.domain,
                    repeat=len(free)
                ):
                    bb = dict(b)
                    bb.update(zip(free, combo))

                    if all(
                        self._neg_holds(c, bb, pos, neg)
                        for c in neg_conds
                    ):
                        out.append(bb)

                return

            pred, args, _ = pos_conds[i]

            for fargs in index.get(pred, ()):
                nb = _rt_unify(args, fargs, b)

                if nb is not None:
                    extend(i + 1, nb)

        extend(0, {})
        return out

    # Optimized _apply method
    def _apply(self, concls, b, pos, neg, index=None):
        changed = False

        for pred, args, is_neg in concls:
            g = (
                pred,
                tuple(b.get(a, a) for a in args)
            )

            if is_neg:
                if self.open_world and g not in neg:
                    neg.add(g)
                    changed = True

            elif g not in pos:
                pos.add(g)

                if index is not None:
                    index[pred].append(g[1])

                changed = True

        return changed

    def _saturate(self, rules, pos, neg):
        any_change = False

        while True:
            changed = False
            index = defaultdict(list)

            for p, a in pos:
                index[p].append(a)

            for conds, concls in rules:
                for b in self._bindings(
                    conds, index, pos, neg
                ):
                    if self._apply(concls, b, pos, neg):
                        changed = True

            if not changed:
                break

            any_change = True

        return any_change

    # Optimized closure method
    def closure(self):
        pos = {
            (p, a)
            for p, a, n in self.facts
            if not n
        }

        neg = {
            (p, a)
            for p, a, n in self.facts
            if n
        }

        pos_rules = [
            r for r in self.rules
            if not any(c[2] for c in r[0])
        ]

        neg_rules = [
            r for r in self.rules
            if any(c[2] for c in r[0])
        ]

        for _ in range(50):
            changed = self._saturate(
                pos_rules, pos, neg
            )

            # Build the positive-fact index once
            index = defaultdict(list)

            for p, a in pos:
                index[p].append(a)

            for conds, concls in neg_rules:
                for b in self._bindings(
                    conds, index, pos, neg
                ):
                    if self._apply(
                        concls, b, pos, neg, index
                    ):
                        changed = True

            if not changed:
                break

        return pos, neg


# Cache parsed contexts
@lru_cache(maxsize=50_000)
def rt_parse_cached(context):
    """Cache parsed facts and rules for repeated contexts."""
    return rt_parse(context)


def rt_answer(context, statement, config=""):
    """Return whether a statement is entailed by the context."""

    facts, rules = rt_parse_cached(context)

    query = rt_clause(statement)

    if not query:
        return False

    pred, args, is_neg = query[0]
    open_world = "owa" in str(config).lower()

    reasoner = RuleTakerReasoner(
        facts,
        rules,
        query_args=args,
        open_world=open_world
    )

    pos, neg = reasoner.closure()

    if is_neg:
        return (
            (pred, args) in neg
            if open_world
            else (pred, args) not in pos
        )

    return (pred, args) in pos


In [14]:
# =====================================================================
# 6. PREDICTION ROUTER (RuleTaker, CounterBench, FOLIO)
# =====================================================================
# Each record is routed by its fields, not its file name.
#   RuleTaker   : context, question (statement), label 'entailment' / 'not entailment'
#   CounterBench: given_info, question, answer
#   FOLIO       : premises, conclusion, label 'True' / 'False' / 'Unknown'

USE_GOLD_FOL = False
FAMILIES = ("RuleTaker", "CounterBench", "FOLIO")


def normalize_gold(value):
    """Map a reference label to True / False / None (None = no usable label)."""
    if value is None:
        return None
    s = str(value).strip().lower()
    if s in ("entailment", "true", "yes", "1"):
        return True
    if s in ("not entailment", "false", "no", "0", "unknown", "uncertain"):
        return False      # FOLIO 'Unknown' counts as not entailed
    return None


def detect_family(record):
    keys = set(record.keys())
    if {"given_info", "question"} <= keys:
        return "CounterBench"
    if {"premises", "conclusion"} <= keys:
        return "FOLIO"
    if "context" in keys and ({"question", "statement", "query"} & keys):
        return "RuleTaker"
    return None


def predict_record(record, fallback_key=0):
    """Returns (record_id, predicted_bool, gold_bool_or_None, family). Raises on bad input."""
    family = detect_family(record)
    record_id = record.get("id", record.get("example_id", fallback_key))

    if family == "CounterBench":
        pred = StructuralModelSolver().compute_counterfactual(record["given_info"], record["question"])
        gold = normalize_gold(record.get("answer"))

    elif family == "FOLIO":
        if USE_GOLD_FOL and "premises-FOL" in record and "conclusion-FOL" in record:
            pred = FolEntailmentChecker.check_entailment(record["premises-FOL"], record["conclusion-FOL"])
        else:
            pred = fol_open_world_entails(record["premises"], record["conclusion"])
        gold = normalize_gold(record.get("label"))

    elif family == "RuleTaker":
        statement = record.get("question", record.get("statement", record.get("query", "")))
        pred = rt_answer(record["context"], statement, record.get("config", ""))
        gold = normalize_gold(record.get("label", record.get("flag")))

    else:
        raise ValueError(f"unrecognised record fields: {sorted(record.keys())}")

    return record_id, bool(pred), gold, family


def evaluate_one(record, fallback_key=0):
    """Like predict_record, but never raises. A failed record is predicted False and its error is kept."""
    try:
        rid, pred, gold, family = predict_record(record, fallback_key)
        return rid, pred, gold, family, None
    except Exception as exc:
        rid = record.get("id", record.get("example_id", fallback_key))
        gold = normalize_gold(record.get("label", record.get("answer", record.get("flag"))))
        family = detect_family(record) or "Unknown"
        return rid, False, gold, family, f"{type(exc).__name__}: {str(exc)[:150]}"


In [15]:

# =====================================================================
# 6.1 SANITY CHECK (Bob example)
# =====================================================================

import os
import json
import csv
import time

sample_reasoner, sample_goal = TextKnowledgeParser.parse_natural_kb(
    "Bob is young.\nBob is kind.\nIf someone is young, then they are smart.\n"
    "If someone is smart and kind, then they are successful.",
    "Bob is successful.",
)

print("=== Sanity Check ===")
print(
    "Query: Bob is successful. -> Prediction:",
    sample_reasoner.infer_forward(sample_goal)
)

for log_line in sample_reasoner.proof_log:
    print("  ", log_line)

print()


# =====================================================================
# 6.2 EVALUATE ALL DATASETS
# Optimized memory usage + accurate error reporting + timing
# =====================================================================

OFFICIAL_INPUT_NAME = None  # e.g. "official_test.jsonl"
# Set this to the official test filename when it is released.

DATA_FOLDER = "Data" if os.path.isdir("Data") else "."

dataset_names = [
    "ruletaker_train.jsonl",
    "ruletaker_test.jsonl",
    "ruletaker_ood_test.jsonl",
    "counterbench_alpha.jsonl",
    "FOLIO_train.jsonl",
    "FOLIO_validation.jsonl",
]

if OFFICIAL_INPUT_NAME and OFFICIAL_INPUT_NAME not in dataset_names:
    dataset_names.append(OFFICIAL_INPUT_NAME)


# ---------------------------------------------------------------------
# Find available datasets
# ---------------------------------------------------------------------

found_names = [
    f for f in dataset_names
    if os.path.exists(os.path.join(DATA_FOLDER, f))
]

for f in dataset_names:
    if f not in found_names:
        print(f"MISSING {os.path.join(DATA_FOLDER, f)}")

if not found_names:
    raise FileNotFoundError(
        f"No dataset files found in '{os.path.abspath(DATA_FOLDER)}'. "
        "Create a Data folder beside the notebook and copy the .jsonl files into it."
    )


# ---------------------------------------------------------------------
# Decide which dataset will be used for submission
# ---------------------------------------------------------------------

if OFFICIAL_INPUT_NAME and OFFICIAL_INPUT_NAME in found_names:
    sub_file = OFFICIAL_INPUT_NAME
else:
    sub_file = found_names[0]
    print(
        f"WARNING: OFFICIAL_INPUT_NAME is not set; "
        f"submission.csv covers only {sub_file}"
    )


# ---------------------------------------------------------------------
# Initialize statistics
# ---------------------------------------------------------------------

family_stats = {
    fam: {
        "records": 0,
        "labelled": 0,
        "correct": 0,
        "wrong": 0,
        "errors": 0,
    }
    for fam in FAMILIES + ("Unknown",)
}

total_records = 0
total_labelled = 0
total_correct = 0
total_wrong = 0
total_errors = 0

dev_rows_count = 0
submission_count = 0

run_start = time.perf_counter()


# ---------------------------------------------------------------------
# Stream predictions directly to CSV files
# This avoids storing hundreds of thousands of dictionaries in memory.
# ---------------------------------------------------------------------

with open(
    "submission.csv",
    "w",
    newline="",
    encoding="utf-8"
) as sub_fh:

    sub_writer = csv.writer(sub_fh)

    sub_writer.writerow(["id", "predicted_label"])

    print(
        f"\n{'File':<28} | {'Records':>7} | {'Labelled':>8} | "
        f"{'Correct':>7} | {'Wrong':>7} | {'Errors':>6} | "
        f"{'Accuracy':>9} | {'Time (s)':>8}"
    )

    print("-" * 112)


    # ================================================================
    # Process each dataset
    # ================================================================

    for fname in found_names:

        path = os.path.join(DATA_FOLDER, fname)

        f_records = 0
        f_labelled = 0
        f_correct = 0
        f_wrong = 0
        f_errors = 0

        error_samples = []

        t0 = time.perf_counter()

        with open(path, "r", encoding="utf-8") as fh:

            for line_no, raw_line in enumerate(fh, start=1):

                raw_line = raw_line.strip()

                if not raw_line:
                    continue

                # ----------------------------------------------------
                # Parse JSON
                # ----------------------------------------------------

                try:
                    record = json.loads(raw_line)

                except json.JSONDecodeError as exc:

                    f_errors += 1

                    if len(error_samples) < 3:
                        error_samples.append(
                            f"line {line_no}: invalid JSON ({exc})"
                        )

                    continue


                # ----------------------------------------------------
                # Run existing reasoning agent
                # ----------------------------------------------------

                rid, pred, gold, family, err = evaluate_one(
                    record,
                    fallback_key=f"{fname}_{line_no}"
                )

                f_records += 1

                fam = family_stats.get(family)

                if fam is None:
                    fam = family_stats["Unknown"]


                pred_label = "True" if pred else "False"

                dev_rows_count += 1


                # ----------------------------------------------------
                # Save official submission predictions
                # only for the selected submission dataset
                # ----------------------------------------------------

                if fname == sub_file:

                    sub_writer.writerow([
                        rid,
                        pred_label
                    ])

                    submission_count += 1


                # ----------------------------------------------------
                # Update family statistics
                # ----------------------------------------------------

                fam["records"] += 1

                # Count reported errors separately
                if err is not None:

                    f_errors += 1
                    fam["errors"] += 1

                    if len(error_samples) < 3:
                        error_samples.append(
                            f"line {line_no}: {err}"
                        )


                # ----------------------------------------------------
                # Evaluate labelled records
                # ----------------------------------------------------

                if gold is not None:

                    f_labelled += 1
                    fam["labelled"] += 1

                    if pred == gold:

                        f_correct += 1
                        fam["correct"] += 1

                    else:

                        f_wrong += 1
                        fam["wrong"] += 1


        # ------------------------------------------------------------
        # Dataset timing and statistics
        # ------------------------------------------------------------

        f_seconds = time.perf_counter() - t0

        acc = (
            f"{f_correct / f_labelled * 100:.2f}%"
            if f_labelled else "N/A"
        )

        print(
            f"{fname:<28} | "
            f"{f_records:>7} | "
            f"{f_labelled:>8} | "
            f"{f_correct:>7} | "
            f"{f_wrong:>7} | "
            f"{f_errors:>6} | "
            f"{acc:>9} | "
            f"{f_seconds:>7.2f}s"
        )

        for msg in error_samples:
            print(f"    Error sample -> {msg}")


# =====================================================================
# 6.3 FINAL ACCURACY BY DATASET FAMILY
# =====================================================================

print("\n=== Accuracy by dataset family ===")

print(
    f"{'Family':<14} | {'Records':>7} | {'Labelled':>8} | "
    f"{'Correct':>7} | {'Wrong':>7} | {'Errors':>6} | {'Accuracy':>9}"
)

print("-" * 86)


for fam in FAMILIES + ("Unknown",):

    s = family_stats[fam]

    if s["records"] == 0:
        continue

    acc = (
        f"{s['correct'] / s['labelled'] * 100:.2f}%"
        if s["labelled"] else "N/A"
    )

    print(
        f"{fam:<14} | "
        f"{s['records']:>7} | "
        f"{s['labelled']:>8} | "
        f"{s['correct']:>7} | "
        f"{s['wrong']:>7} | "
        f"{s['errors']:>6} | "
        f"{acc:>9}"
    )

    total_records += s["records"]
    total_labelled += s["labelled"]
    total_correct += s["correct"]
    total_wrong += s["wrong"]
    total_errors += s["errors"]


# =====================================================================
# 6.4 OVERALL RESULTS
# =====================================================================

print("-" * 86)

overall_acc = (
    f"{total_correct / total_labelled * 100:.2f}%"
    if total_labelled else "N/A"
)

print(
    f"{'OVERALL':<14} | "
    f"{total_records:>7} | "
    f"{total_labelled:>8} | "
    f"{total_correct:>7} | "
    f"{total_wrong:>7} | "
    f"{total_errors:>6} | "
    f"{overall_acc:>9}"
)


# =====================================================================
# 6.5 FINAL INFERENCE TIME
# =====================================================================

run_seconds = time.perf_counter() - run_start

ms_per_query = (
    run_seconds * 1000 / total_records
    if total_records else 0
)

print("\n=== Runtime Summary ===")

print(f"Total records processed: {total_records}")
print(f"Total inference time:    {run_seconds:.2f} seconds")
print(f"Average time per query:  {ms_per_query:.4f} ms")

print("\n=== Prediction Summary ===")

# print(f"Development predictions saved: {dev_rows_count}")
# print(f"Submission predictions saved:  {submission_count}")
# print(f"Submission dataset:            {sub_file}")

print(f"\nSaved submission.csv with {submission_count} predictions ({sub_file}).")

# print("\nNote:")
# print("- Wrong = incorrect predictions.")
# print("- Errors = invalid JSON or errors reported by evaluate_one().")
# print("- Labelled = records with a usable reference label.")


=== Sanity Check ===
Query: Bob is successful. -> Prediction: True
   Initial Facts: {kind(bob), young(bob)}
   Derived smart(bob) from young(bob) via (young(?x) → smart(?x))
   Derived successful(bob) from smart(bob) ∧ kind(bob) via (smart(?x) ∧ kind(?x) → successful(?x))


File                         | Records | Labelled | Correct |   Wrong | Errors |  Accuracy | Time (s)
----------------------------------------------------------------------------------------------------------------
ruletaker_train.jsonl        |  480152 |   480152 |  470263 |    9889 |      0 |    97.94% |  154.04s
ruletaker_test.jsonl         |  151911 |   151911 |  148578 |    3333 |      0 |    97.81% |   49.80s
ruletaker_ood_test.jsonl     |     375 |      375 |     373 |       2 |      0 |    99.47% |    0.19s
counterbench_alpha.jsonl     |    1000 |     1000 |     960 |      40 |      0 |    96.00% |    0.14s
FOLIO_train.jsonl            |    1001 |     1001 |     644 |     357 |      0 |    64.34% |    1.22s

## 7. Experimental Results

The system was evaluated on six dataset splits covering RuleTaker, CounterBench, and FOLIO.

| Dataset | Number of Queries | Correct Predictions | Accuracy |
|---|---:|---:|---:|
| `ruletaker_train` | 480,152 | 470,263 | 97.94% |
| `ruletaker_test` | 151911 | 148578 | 97.81% |
| `ruletaker_ood_test` | 375 | 373 | 99.47% |
| `counterbench_alpha` | 1,000 | 960 | 96.00% |
| `FOLIO_train` | 1,001 | 644 | 64.34% |
| `FOLIO_validation` | 203 | 136 | 67.00% |
| **Overall** | **634,642** | **620,954** | **97.84%** |


### Key Findings

- **RuleTaker:** 97.94% on train, 97.81% on test and 99.47% on the out-of-distribution test set.
- **CounterBench:** The system achieved 96% overall accuracy, with performance varying across reasoning categories: basic, 250/250; conditional, 230/250; joint, 240/250; and nested, 240/250. Basic reasoning achieved perfect accuracy, while conditional reasoning was comparatively more challenging
- **FOLIO:** English premises are converted to first-order logic by a rule-based parser and decided by a resolution prover. The gold FOL annotations are not used (`USE_GOLD_FOL = False`). Accuracy is 64.34% on train and 67.00% on validation against always-False baselines of 61.2% and 64.5%. With the gold annotations the earlier DPLL pipeline reached 77.32% and 81.28%, which is not a fair estimate for unseen knowledge bases.

### Limitations and Error Analysis

The current system has several constraints:

1. **Binary predictions:** The system uses a closed-world assumption, treating conclusions that cannot be proved as False. This does not explicitly distinguish between false and uncertain conclusions.
2. **Bounded FOL reasoning:**The grounding process for quantifiers and the 100-node limit in the DPLL solver may prevent the prover from establishing some valid entailments.

3. **Causal assumptions:** Unobserved root variables default to `False`.
4. **Incomplete causal observations:** Observations of derived variables are not incorporated into the causal equations.
5. **FOLIO evaluation:**The system produces only binary predictions and does not explicitly predict Uncertain. Consequently, under strict three-class evaluation, its accuracy is approximately 46% on the training split and 50% on the validation split.

Overall, the system demonstrates strong performance on RuleTaker and CounterBench. However, FOLIO remains more challenging because of its first-order logic requirements, uncertainty handling, and the search limitations of the bounded theorem prover.

# Test Cell

### Converting .csv to .jsonl

In [ ]:
import csv, json

CSV_PATH = "private_test.csv"      # input
JSONL_PATH = "data\private_test.jsonl"  # output
DATASET, SPLIT = "private_test", "test"

def to_bool(x):
    return str(x).strip().lower() == "true"

def lines(s):
    # one formula per context sentence (multi-line cell -> list)
    return [ln.strip() for ln in s.splitlines() if ln.strip()]

n = 0
with open(CSV_PATH, newline="", encoding="utf-8") as f, \
     open(JSONL_PATH, "w", encoding="utf-8") as out:
    for row in csv.DictReader(f):
        stmt_logic = row["statement_logic"].strip()
        ex = {
            "id": row["id"].strip(),
            "context": row["context"].strip(),
            "query": row["statement"].strip(),          # 'statement' -> 'query'
            "label": to_bool(row["label"]),             # True iff entailed
            "dataset": DATASET,
            "split": SPLIT,
            "meta": {
                "raw_label": row["label"].strip(),
                "style": "counterfactual" if "_{" in stmt_logic else "entailment",
                # gold logic: for checking the parser only, never at prediction time
                "premises_fol": lines(row["context_logic"]),
                "conclusion_fol": stmt_logic,
            },
        }
        out.write(json.dumps(ex, ensure_ascii=False) + "\n")
        n += 1

print(f"Wrote {n} examples to {JSONL_PATH}")

In [20]:


ood_file = os.path.join(DATA_FOLDER, "private_test.jsonl")

wrong = []

with open(ood_file, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        record = json.loads(line)
        rid, pred, gold, family, error = evaluate_one(record, i)

        if error or (gold is not None and pred != gold):
            wrong.append({
                "id": rid,
                "context": record.get("context"),
                "statement": record.get("statement"),
                "gold": gold,
                "predicted": pred,
                "config": record.get("config"),
                "error": error
            })

print("Wrong predictions:", len(wrong))

for item in wrong[:10]:
    print("\n" + "=" * 60)
    for key, value in item.items():
        print(f"{key}: {value}")

Wrong predictions: 40

id: P003
context: We know that Vapo causes Toln, Drav causes Toln, Toln causes Jask, Toln causes Mora, Toln causes Huld, and Mora causes Fesh. Nothing else causes any of these events. We observed Drav. Vapo did not occur.
statement: None
gold: True
predicted: False
config: None
error: None

id: P005
context: Every robot built by Kestrel Labs runs on solar power or on batteries. Robots that run on solar power cannot operate at night. Unit Vex was built by Kestrel Labs. Unit Vex operates at night.
statement: None
gold: True
predicted: False
config: None
error: None

id: P006
context: Every robot built by Kestrel Labs runs on solar power or on batteries. Robots that run on solar power cannot operate at night. Unit Vex was built by Kestrel Labs. Unit Vex operates at night.
statement: None
gold: True
predicted: False
config: None
error: None

id: P009
context: Every member of the Quorvan choir sings either soprano or alto. Anyone who sings soprano rehearses on Tuesday

In [21]:
import time
import json
import os

ood_file = os.path.join(DATA_FOLDER, "private_test.jsonl")

wrong = []
query_times = []

correct = 0
scored = 0
total_inference_time = 0.0
total_queries = 0

with open(ood_file, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        record = json.loads(line)

        start = time.perf_counter()
        rid, pred, gold, family, error = evaluate_one(record, i)
        elapsed = time.perf_counter() - start

        total_inference_time += elapsed
        total_queries += 1

        query_times.append({
            "id": rid,
            "time_ms": elapsed * 1000
        })

        if gold is not None:
            scored += 1
            if not error and pred == gold:
                correct += 1

        if error or (gold is not None and pred != gold):
            wrong.append({
                "id": rid,
                "context": record.get("context"),
                "statement": record.get("statement"),
                "gold": gold,
                "predicted": pred,
                "config": record.get("config"),
                "error": error,
                "inference_time_ms": elapsed * 1000
            })

accuracy = (correct / scored * 100) if scored else 0
average_time_ms = (
    total_inference_time / total_queries * 1000
    if total_queries else 0
)

print(f"Total queries: {total_queries}")
print(f"Total inference time: {total_inference_time:.4f} seconds")
print(f"Average inference time per query: {average_time_ms:.3f} ms")
print(f"Correct predictions: {correct}")
print(f"Scored queries: {scored}")
print(f"Accuracy: {accuracy:.2f}%")
print(f"Wrong predictions: {len(wrong)}")



Total queries: 100
Total inference time: 0.0242 seconds
Average inference time per query: 0.242 ms
Correct predictions: 60
Scored queries: 100
Accuracy: 60.00%
Wrong predictions: 40
